In [10]:

# Prepare a Mandarin Dataset for Human Review


from pathlib import Path
import json
import hashlib
import random
import sys
from datetime import datetime, timezone

print("=" * 70)
print("TASK A: PREPARE A MANDARIN DATASET FOR HUMAN REVIEW")
print("=" * 70)


# ------------------------------------------------------------

SEED = 42
random.seed(SEED)



DIRECTIONS = {
    "en-zh": {
        "source_language": "English",
        "target_language": "Mandarin Chinese",
    },
    "zh-en": {
        "source_language": "Mandarin Chinese",
        "target_language": "English",
    },
}



TARGET_PER_DIRECTION = 100

TOTAL_TARGET = TARGET_PER_DIRECTION * len(DIRECTIONS)



QUALITY_TARGETS = {
    "high": 0.25,
    "medium": 0.25,
    "low": 0.25,
    "very_low": 0.25,
}

assert abs(sum(QUALITY_TARGETS.values()) - 1.0) < 1e-9


DATASET_NAME = "mandarin_human_review"
DATASET_VERSION = "v1.0"

CREATED_AT = datetime.now(timezone.utc).isoformat()



OUTPUT_DIR = Path("./task_a_human_review")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REVIEW_FILE = OUTPUT_DIR / "human_review_items.csv"
PROVENANCE_FILE = OUTPUT_DIR / "provenance.jsonl"
INSTRUCTIONS_FILE = OUTPUT_DIR / "reviewer_instructions.md"
MANIFEST_FILE = OUTPUT_DIR / "dataset_manifest.json"
VALIDATION_FILE = OUTPUT_DIR / "validation_report.json"



DATA_SEPARATION_POLICY = {
    "reuse_finetuning_examples": False,
    "reuse_existing_judge_calibration_examples": False,
    "reuse_prompt_validation_examples": False,
    "reuse_sealed_final_test_examples": False,
}

# ------------------------------------------------------------
# 8. CONFIGURATION MANIFEST
# ------------------------------------------------------------

TASK_CONFIG = {
    "task": "Prepare A Mandarin Dataset For Human Review",
    "dataset_name": DATASET_NAME,
    "dataset_version": DATASET_VERSION,
    "created_at_utc": CREATED_AT,
    "seed": SEED,
    "directions": DIRECTIONS,
    "target_per_direction": TARGET_PER_DIRECTION,
    "total_target": TOTAL_TARGET,
    "quality_targets": QUALITY_TARGETS,
    "data_separation_policy": DATA_SEPARATION_POLICY,
}

# Stable hash of configuration
config_string = json.dumps(
    TASK_CONFIG,
    sort_keys=True,
    ensure_ascii=False
).encode("utf-8")

CONFIG_SHA256 = hashlib.sha256(config_string).hexdigest()

TASK_CONFIG["config_sha256"] = CONFIG_SHA256

print("\n✓ Task configuration created")
print(f"✓ Dataset: {DATASET_NAME}")
print(f"✓ Version: {DATASET_VERSION}")
print(f"✓ Random seed: {SEED}")

print("\nTarget examples:")
for direction in DIRECTIONS:
    print(f"  {direction}: {TARGET_PER_DIRECTION}")

print(f"\n✓ Total target examples: {TOTAL_TARGET}")

print("\nInternal quality distribution:")
for quality, proportion in QUALITY_TARGETS.items():
    expected = int(TARGET_PER_DIRECTION * proportion)
    print(
        f"  {quality:<10} "
        f"{proportion:.0%} "
        f"(~{expected} per direction)"
    )

print("\nData separation:")
for key, value in DATA_SEPARATION_POLICY.items():
    print(f"  {key}: {value}")

print(f"\n✓ Output directory: {OUTPUT_DIR.resolve()}")
print(f"✓ Config SHA256: {CONFIG_SHA256}")



TASK A: PREPARE A MANDARIN DATASET FOR HUMAN REVIEW

✓ Task configuration created
✓ Dataset: mandarin_human_review
✓ Version: v1.0
✓ Random seed: 42

Target examples:
  en-zh: 100
  zh-en: 100

✓ Total target examples: 200

Internal quality distribution:
  high       25% (~25 per direction)
  medium     25% (~25 per direction)
  low        25% (~25 per direction)
  very_low   25% (~25 per direction)

Data separation:
  reuse_finetuning_examples: False
  reuse_existing_judge_calibration_examples: False
  reuse_prompt_validation_examples: False
  reuse_sealed_final_test_examples: False

✓ Output directory: /content/task_a_human_review
✓ Config SHA256: d9bb0f89f6745cfd42a6a452d781b88a8f351e7c50753973144a1d6c12cae736


In [11]:

# : DEFINE INDEPENDENT SOURCE + PROVENANCE SCHEMA




EXCLUDED_SOURCES = {
    "RicardoRei/wmt-sqm-human-evaluation",
    "Helsinki-NLP/opus-100",
}

print("\nExcluded borrowed sources:")
for source in sorted(EXCLUDED_SOURCES):
    print(f"  - {source}")




SOURCE_CONFIG = {
    "dataset_name": "FradSer/OpenSubtitles-en-zh-cn-20m",
    "dataset_config": None,
    "source_type": "independent_bilingual_subtitle_corpus",
    "intended_use": "source/reference pool for Task A",
    "original_borrowed_source": False,
}

NEW_SOURCE_NAME = SOURCE_CONFIG["dataset_name"]

# Guard against accidentally selecting an excluded source.
assert NEW_SOURCE_NAME not in EXCLUDED_SOURCES, (
    "ERROR: Task A source overlaps with an excluded "
    "borrowed judge-calibration source."
)

print("\n✓ Independent Task A source selected")
print(f"  Dataset: {SOURCE_CONFIG['dataset_name']}")
print(f"  Config:  {SOURCE_CONFIG['dataset_config']}")
print(f"  Type:    {SOURCE_CONFIG['source_type']}")




REQUIRED_PROVENANCE_FIELDS = [
    "example_id",
    "direction",
    "source_dataset",
    "source_config",
    "source_split",
    "source_record_id",
    "source_language",
    "target_language",
    "original_source_text",
    "original_reference_text",
    "candidate_generation_method",
    "internal_quality_bucket",
    "dataset_version",
]

print("\nRequired provenance fields:")
for field in REQUIRED_PROVENANCE_FIELDS:
    print(f"  ✓ {field}")




REVIEWER_VISIBLE_FIELDS = [
    "example_id",
    "direction",
    "source_text",
    "candidate_translation",
    "reference_translation",
    "human_score",
    "reviewer_comment",
]

HIDDEN_FROM_REVIEWER = sorted(
    set(REQUIRED_PROVENANCE_FIELDS)
    - set(REVIEWER_VISIBLE_FIELDS)
)

print("\nReviewer-visible fields:")
for field in REVIEWER_VISIBLE_FIELDS:
    print(f"  ✓ {field}")

print("\nInternal metadata hidden from reviewer:")
for field in HIDDEN_FROM_REVIEWER:
    print(f"  - {field}")



TRACEABILITY_POLICY = {
    "record_dataset_name": True,
    "record_dataset_config": True,
    "record_source_split": True,
    "record_source_record_id": True,
    "record_original_text": True,
    "record_reference_text": True,
    "record_generation_method": True,
    "record_internal_quality_bucket": True,
}

assert all(TRACEABILITY_POLICY.values()), (
    "All traceability requirements must be enabled."
)




SOURCE_INDEPENDENCE_POLICY = {
    "exclude_existing_wmt_sqm": True,
    "new_examples_required": True,
    "preserve_source_record_ids": True,
    "preserve_original_bilingual_pair": True,
}

assert all(SOURCE_INDEPENDENCE_POLICY.values())




TASK_CONFIG["source_config"] = SOURCE_CONFIG
TASK_CONFIG["excluded_sources"] = sorted(EXCLUDED_SOURCES)
TASK_CONFIG["required_provenance_fields"] = REQUIRED_PROVENANCE_FIELDS
TASK_CONFIG["reviewer_visible_fields"] = REVIEWER_VISIBLE_FIELDS
TASK_CONFIG["traceability_policy"] = TRACEABILITY_POLICY
TASK_CONFIG["source_independence_policy"] = SOURCE_INDEPENDENCE_POLICY




print("\n" + "-" * 70)
print("SOURCE POLICY SUMMARY")
print("-" * 70)

print(f"""
Original borrowed judge data:
    RicardoRei/wmt-sqm-human-evaluation

Task A source:
    {SOURCE_CONFIG['dataset_name']}
    config = {SOURCE_CONFIG['dataset_config']}

Target:
    EN → ZH : {TARGET_PER_DIRECTION}
    ZH → EN : {TARGET_PER_DIRECTION}
    TOTAL   : {TOTAL_TARGET}

Rule:
    Task A examples must be newly selected from the independent
    source and must retain provenance information.
""")

print("✓ Independent-source policy defined")
print("✓ Provenance schema defined")
print("✓ Reviewer/internal metadata separation defined")




Excluded borrowed sources:
  - Helsinki-NLP/opus-100
  - RicardoRei/wmt-sqm-human-evaluation

✓ Independent Task A source selected
  Dataset: FradSer/OpenSubtitles-en-zh-cn-20m
  Config:  None
  Type:    independent_bilingual_subtitle_corpus

Required provenance fields:
  ✓ example_id
  ✓ direction
  ✓ source_dataset
  ✓ source_config
  ✓ source_split
  ✓ source_record_id
  ✓ source_language
  ✓ target_language
  ✓ original_source_text
  ✓ original_reference_text
  ✓ candidate_generation_method
  ✓ internal_quality_bucket
  ✓ dataset_version

Reviewer-visible fields:
  ✓ example_id
  ✓ direction
  ✓ source_text
  ✓ candidate_translation
  ✓ reference_translation
  ✓ human_score
  ✓ reviewer_comment

Internal metadata hidden from reviewer:
  - candidate_generation_method
  - dataset_version
  - internal_quality_bucket
  - original_reference_text
  - original_source_text
  - source_config
  - source_dataset
  - source_language
  - source_record_id
  - source_split
  - target_language

-

In [12]:

#  LOAD + INSPECT INDEPENDENT BILINGUAL SOURCE




# ------------------------------------------------------------

try:
    from datasets import load_dataset
    print("✓ Hugging Face datasets imported")
except ImportError:
    print("Installing datasets...")
    !pip -q install datasets
    from datasets import load_dataset
    print("✓ Hugging Face datasets installed and imported")




DATASET_ID = SOURCE_CONFIG["dataset_name"]
DATASET_CONFIG = SOURCE_CONFIG["dataset_config"]

print("\nLoading independent dataset...")
print(f"  Dataset: {DATASET_ID}")
print(f"  Config:  {DATASET_CONFIG}")

try:
    if DATASET_CONFIG is None:
      independent_dataset = load_dataset(
          DATASET_ID
      )
    else:
      independent_dataset = load_dataset(
          DATASET_ID,
          DATASET_CONFIG,
      )

    print("\n✓ Dataset loaded successfully")

except Exception as e:
    print("\n✗ DATASET LOAD FAILED")
    print(f"Error type: {type(e).__name__}")
    print(f"Error: {e}")

    independent_dataset = None




if independent_dataset is None:

    print("\n" + "-" * 70)
    print("The configured source could not be loaded.")
    print("Do NOT continue to sampling.")
    print("We will select another independent source if necessary.")
    print("-" * 70)

else:


    print("\nAvailable splits:")

    for split_name in independent_dataset.keys():

        split_data = independent_dataset[split_name]

        print(
            f"  ✓ {split_name:<15} "
            f"{len(split_data):,} records"
        )




    available_splits = list(independent_dataset.keys())

    if "train" in available_splits:
        inspection_split = "train"
    else:
        inspection_split = available_splits[0]

    inspection_data = independent_dataset[inspection_split]

    print(
        f"\nInspection split: {inspection_split}"
    )




    print("\nDataset features:")
    print(inspection_data.features)

    print("\nColumn names:")
    for column in inspection_data.column_names:
        print(f"  - {column}")



    print("\n" + "-" * 70)
    print("FIRST 3 RAW RECORDS")
    print("-" * 70)

    number_to_show = min(3, len(inspection_data))

    for i in range(number_to_show):

        print(f"\nRecord {i}:")
        print(
            json.dumps(
                inspection_data[i],
                ensure_ascii=False,
                indent=2,
                default=str,
            )
        )


    # --------------------------------------------------------
    # 8. BASIC DATASET INFORMATION
    # --------------------------------------------------------

    SOURCE_INSPECTION = {
        "dataset_name": DATASET_ID,
        "dataset_config": DATASET_CONFIG,
        "available_splits": available_splits,
        "inspection_split": inspection_split,
        "num_records_in_inspection_split": len(
            inspection_data
        ),
        "column_names": list(
            inspection_data.column_names
        ),
        "features": str(
            inspection_data.features
        ),
    }

    TASK_CONFIG["source_inspection"] = SOURCE_INSPECTION



    minimum_needed = TARGET_PER_DIRECTION

    print("\n" + "-" * 70)
    print("CAPACITY CHECK")
    print("-" * 70)

    print(
        f"Records available: {len(inspection_data):,}"
    )

    print(
        f"Minimum base records needed: "
        f"{minimum_needed}"
    )

    if len(inspection_data) >= minimum_needed:

        print(
            "✓ Dataset has enough raw records "
            "for the current target."
        )

    else:

        print(
            "✗ Dataset does NOT contain enough "
            "records for the target."
        )


    # --------------------------------------------------------
    # 10. IMPORTANT: NO SAMPLING YET
    # --------------------------------------------------------

    print("\nNOTE:")
    print(
        "No Task A examples have been selected yet."
    )
    print(
        "This cell only loads and inspects the "
        "independent source."
    )



✓ Hugging Face datasets imported

Loading independent dataset...
  Dataset: FradSer/OpenSubtitles-en-zh-cn-20m
  Config:  None

✓ Dataset loaded successfully

Available splits:
  ✓ train           19,628,420 records

Inspection split: train

Dataset features:
{'source': Value('string'), 'target': Value('string')}

Column names:
  - source
  - target

----------------------------------------------------------------------
FIRST 3 RAW RECORDS
----------------------------------------------------------------------

Record 0:
{
  "source": "Previously on \"The Blacklist\"...",
  "target": "前情提要"
}

Record 1:
{
  "source": "- You want to call your daddy?",
  "target": "- 想给爸爸打电话吗?"
}

Record 2:
{
  "source": "- Yeah, I want to tell him I'm okay.",
  "target": "- 嗯 我想告诉他我没事"
}

----------------------------------------------------------------------
CAPACITY CHECK
----------------------------------------------------------------------
Records available: 19,628,420
Minimum base records needed: 100

In [13]:

# CLEAN + VALIDATE INDEPENDENT BILINGUAL SOURCE POOL



import re
import random
import unicodedata
from collections import Counter



SOURCE_SPLIT = "train"
source_data = independent_dataset[SOURCE_SPLIT]

print(f"\nSource split: {SOURCE_SPLIT}")
print(f"Total source records: {len(source_data):,}")




MAX_RECORDS_TO_INSPECT = 20_000
ELIGIBLE_POOL_TARGET = 2_000

print("\nPool construction configuration:")
print(f"  Maximum records to inspect: {MAX_RECORDS_TO_INSPECT:,}")
print(f"  Eligible pool target:       {ELIGIBLE_POOL_TARGET:,}")
print(f"  Random seed:                {SEED}")



def remove_invisible_unicode(text):
    """
    Remove invisible Unicode formatting/control characters
    commonly found in subtitle corpora.
    """

    if text is None:
        return ""

    text = str(text)

    # Explicit characters commonly observed in subtitles.
    invisible_chars = [
        "\u200b",  # zero-width space
        "\u200c",  # zero-width non-joiner
        "\u200d",  # zero-width joiner
        "\u200e",  # left-to-right mark
        "\u200f",  # right-to-left mark
        "\u202a",  # left-to-right embedding
        "\u202b",  # right-to-left embedding
        "\u202c",  # pop directional formatting
        "\u202d",  # left-to-right override
        "\u202e",  # right-to-left override
        "\u2060",  # word joiner
        "\ufeff",  # BOM / zero-width no-break space
    ]

    for char in invisible_chars:
        text = text.replace(char, "")

    # Remove remaining Unicode formatting characters (Cf).
    text = "".join(
        char
        for char in text
        if unicodedata.category(char) != "Cf"
    )

    return text


def normalize_whitespace(text):
    """
    Collapse repeated whitespace and strip edges.
    """

    if text is None:
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(text)
    ).strip()


def clean_text(text):
    """
    Apply the standard Task A text cleaning pipeline.
    """

    text = remove_invisible_unicode(text)
    text = normalize_whitespace(text)

    return text


# ------------------------------------------------------------
# 4. LANGUAGE / LENGTH HELPERS
# ------------------------------------------------------------

def contains_chinese(text):
    """
    Return True when at least one CJK character is present.
    """

    return bool(
        re.search(
            r"[\u3400-\u4DBF\u4E00-\u9FFF]",
            text
        )
    )


def chinese_char_count(text):
    """
    Count Chinese/CJK characters.
    """

    return len(
        re.findall(
            r"[\u3400-\u4DBF\u4E00-\u9FFF]",
            text
        )
    )


def latin_letter_count(text):
    """
    Count Latin alphabetic characters.
    """

    return len(
        re.findall(
            r"[A-Za-z]",
            text
        )
    )


def token_count_english(text):
    """
    Approximate English token count.
    """

    return len(
        re.findall(
            r"[A-Za-z0-9]+(?:['’-][A-Za-z0-9]+)*",
            text
        )
    )


def latin_ratio(text):
    """
    Calculate the proportion of Latin letters relative to
    Latin + Chinese characters.

    Used to detect substantial English contamination on the
    Chinese side of an aligned bilingual pair.
    """

    latin = latin_letter_count(text)
    chinese = chinese_char_count(text)

    total = latin + chinese

    if total == 0:
        return 0.0

    return latin / total


# ------------------------------------------------------------
# 5. SOURCE-COPY DETECTION
# ------------------------------------------------------------

def normalize_for_comparison(text):
    """
    Normalize text for contamination/source-copy checks.
    """

    text = clean_text(text)
    text = text.casefold()

    # Normalize spacing around punctuation.
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def source_copied_into_target(en, zh):
    """
    Detect obvious cases where the English source appears
    verbatim inside the Chinese target.

    Example rejected:

        EN:
        I'm gonna kill them all.

        ZH:
        I'm gonna kill them all. 我要把他们都杀了。
    """

    en_compare = normalize_for_comparison(en)
    zh_compare = normalize_for_comparison(zh)

    # Avoid over-triggering on tiny strings.
    if len(en_compare) < 10:
        return False

    return en_compare in zh_compare



def validate_pair(source_text, target_text):

    en = clean_text(source_text)
    zh = clean_text(target_text)

    # --------------------------------------------------------
    # Empty records
    # --------------------------------------------------------

    if not en or not zh:
        return False, "empty"

    # --------------------------------------------------------
    # Length
    # --------------------------------------------------------

    en_tokens = token_count_english(en)
    zh_chars = chinese_char_count(zh)

    if en_tokens < 4:
        return False, "english_too_short"

    if zh_chars < 4:
        return False, "chinese_too_short"

    if en_tokens > 40:
        return False, "english_too_long"

    if zh_chars > 80:
        return False, "chinese_too_long"

    # --------------------------------------------------------
    # Language sanity
    # --------------------------------------------------------

    if latin_letter_count(en) < 4:
        return False, "english_language_check"

    if not contains_chinese(zh):
        return False, "chinese_language_check"



    if latin_ratio(zh) > 0.30:
        return False, "chinese_english_contamination"

    # --------------------------------------------------------
    # Source copied into target
    # --------------------------------------------------------

    if source_copied_into_target(en, zh):
        return False, "source_copied_into_target"

    # --------------------------------------------------------
    # Subtitle markup / metadata
    # --------------------------------------------------------

    markup_patterns = [
        r"<[^>]+>",
        r"\{\\[^}]+\}",
        r"^\s*\[.*\]\s*$",
        r"^\s*\(.*\)\s*$",
    ]

    for pattern in markup_patterns:

        if re.search(pattern, en):
            return False, "english_markup"

        if re.search(pattern, zh):
            return False, "chinese_markup"

    # --------------------------------------------------------
    # URLs
    # --------------------------------------------------------

    if re.search(
        r"https?://|www\.",
        en,
        flags=re.IGNORECASE
    ):
        return False, "url"

    if re.search(
        r"https?://|www\.",
        zh,
        flags=re.IGNORECASE
    ):
        return False, "url"

    return True, "eligible"



print("\nCreating deterministic source indices...")

inspection_limit = min(
    MAX_RECORDS_TO_INSPECT,
    len(source_data)
)

pool_rng = random.Random(SEED)

search_indices = pool_rng.sample(
    range(len(source_data)),
    inspection_limit
)

assert len(search_indices) == inspection_limit
assert len(search_indices) == len(set(search_indices))

print(
    f"✓ Generated {len(search_indices):,} "
    "deterministic original row indices"
)

print("✓ All sampled row indices are unique")




eligible_pool = []

rejection_counts = Counter()

seen_pairs = set()

records_inspected = 0


print("\nScanning and validating source records...")


for source_record_id in search_indices:

    row = source_data[int(source_record_id)]

    records_inspected += 1

    # Clean both sides before validation/storage.
    en = clean_text(
        row.get("source", "")
    )

    zh = clean_text(
        row.get("target", "")
    )

    valid, reason = validate_pair(
        en,
        zh
    )

    if not valid:

        rejection_counts[reason] += 1
        continue



    pair_key = (
        en.casefold(),
        zh
    )

    if pair_key in seen_pairs:

        rejection_counts["duplicate_pair"] += 1
        continue

    seen_pairs.add(pair_key)

    # --------------------------------------------------------
    # Store clean pair + provenance
    # --------------------------------------------------------

    eligible_pool.append(
        {
            "source_record_id": int(
                source_record_id
            ),

            "source_dataset": DATASET_ID,

            "source_config": DATASET_CONFIG,

            "source_split": SOURCE_SPLIT,

            "english": en,

            "chinese": zh,

            "english_token_count": (
                token_count_english(en)
            ),

            "chinese_char_count": (
                chinese_char_count(zh)
            ),

            "chinese_latin_ratio": (
                latin_ratio(zh)
            ),
        }
    )

    # Stop once the requested clean pool has been reached.

    if len(eligible_pool) >= ELIGIBLE_POOL_TARGET:
        break


# ------------------------------------------------------------
# 9. POOL RESULTS
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("POOL CONSTRUCTION RESULTS")
print("-" * 70)

print(
    f"Records inspected: "
    f"{records_inspected:,}"
)

print(
    f"Eligible records:  "
    f"{len(eligible_pool):,}"
)

acceptance_rate = (
    len(eligible_pool) / records_inspected
    if records_inspected
    else 0.0
)

print(
    f"Acceptance rate:   "
    f"{acceptance_rate:.1%}"
)


# ------------------------------------------------------------
# 10. REJECTION BREAKDOWN
# ------------------------------------------------------------

print("\nRejected records:")

if rejection_counts:

    for reason, count in rejection_counts.most_common():

        print(
            f"  {reason:<32} "
            f"{count:,}"
        )

else:

    print("  None")


# ------------------------------------------------------------
# 11. CAPACITY CHECKS
# ------------------------------------------------------------

assert len(eligible_pool) >= TOTAL_TARGET, (
    f"Need at least {TOTAL_TARGET} clean records "
    f"for Task A, but only {len(eligible_pool)} "
    "were found."
)

print(
    f"\n✓ Pool contains enough clean records "
    f"for all {TOTAL_TARGET} final Task A examples"
)


if len(eligible_pool) >= ELIGIBLE_POOL_TARGET:

    print(
        f"✓ Eligible pool target reached: "
        f"{ELIGIBLE_POOL_TARGET:,}"
    )

else:

    print(
        f"⚠ Pool target of {ELIGIBLE_POOL_TARGET:,} "
        "was not reached."
    )


# ------------------------------------------------------------
# 12. DUPLICATE VALIDATION
# ------------------------------------------------------------

unique_pairs = {
    (
        row["english"].casefold(),
        row["chinese"]
    )
    for row in eligible_pool
}

assert len(unique_pairs) == len(eligible_pool), (
    "Duplicate bilingual pairs remain."
)

print("✓ No duplicate bilingual pairs")


# ------------------------------------------------------------
# 13. SOURCE-ID VALIDATION
# ------------------------------------------------------------

source_record_ids = [
    row["source_record_id"]
    for row in eligible_pool
]

assert (
    len(source_record_ids)
    == len(set(source_record_ids))
), "Duplicate source record IDs detected."

assert all(
    0 <= record_id < len(source_data)
    for record_id in source_record_ids
), "Invalid source record ID detected."

print("✓ Source record IDs are unique")
print("✓ Source record IDs point to original train rows")


# ------------------------------------------------------------
# 14. CONTAMINATION VALIDATION
# ------------------------------------------------------------

assert all(
    not source_copied_into_target(
        row["english"],
        row["chinese"]
    )
    for row in eligible_pool
), "Source-copy contamination remains."

assert all(
    row["chinese_latin_ratio"] <= 0.30
    for row in eligible_pool
), "English-contaminated Chinese records remain."

print("✓ No source-copy contamination detected")
print("✓ Chinese-side English contamination check passed")


# ------------------------------------------------------------
# 15. INVISIBLE UNICODE VALIDATION
# ------------------------------------------------------------

def has_format_character(text):

    return any(
        unicodedata.category(char) == "Cf"
        for char in text
    )


assert all(
    not has_format_character(row["english"])
    and not has_format_character(row["chinese"])
    for row in eligible_pool
), "Invisible Unicode formatting characters remain."

print("✓ Invisible Unicode formatting characters removed")


# ------------------------------------------------------------
# 16. LANGUAGE VALIDATION
# ------------------------------------------------------------

assert all(
    latin_letter_count(row["english"]) >= 4
    for row in eligible_pool
)

assert all(
    contains_chinese(row["chinese"])
    for row in eligible_pool
)

print("✓ Basic EN/ZH language checks passed")


# ------------------------------------------------------------
# 17. DISPLAY CLEAN SAMPLE
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("SAMPLE CLEAN ELIGIBLE PAIRS")
print("-" * 70)

sample_count = min(
    5,
    len(eligible_pool)
)

for i, row in enumerate(
    eligible_pool[:sample_count],
    start=1
):

    print(f"\nExample {i}")

    print(
        f"Source record ID: "
        f"{row['source_record_id']}"
    )

    print(
        f"EN: {row['english']}"
    )

    print(
        f"ZH: {row['chinese']}"
    )

    print(
        "Lengths: "
        f"{row['english_token_count']} EN tokens / "
        f"{row['chinese_char_count']} Chinese chars"
    )

    print(
        f"Chinese Latin ratio: "
        f"{row['chinese_latin_ratio']:.3f}"
    )


# ------------------------------------------------------------
# 18. SAVE POOL METADATA
# ------------------------------------------------------------

POOL_INFO = {
    "source_dataset": DATASET_ID,

    "source_config": DATASET_CONFIG,

    "source_split": SOURCE_SPLIT,

    "sampling_method": (
        "deterministic_random_original_row_indices"
    ),

    "seed": SEED,

    "max_records_to_inspect": (
        MAX_RECORDS_TO_INSPECT
    ),

    "records_inspected": (
        records_inspected
    ),

    "eligible_pool_target": (
        ELIGIBLE_POOL_TARGET
    ),

    "eligible_records": (
        len(eligible_pool)
    ),

    "acceptance_rate": (
        acceptance_rate
    ),

    "rejection_counts": (
        dict(rejection_counts)
    ),

    "filters": {
        "minimum_english_tokens": 4,
        "maximum_english_tokens": 40,
        "minimum_chinese_characters": 4,
        "maximum_chinese_characters": 80,
        "maximum_chinese_latin_ratio": 0.30,
        "reject_source_copy": True,
        "remove_invisible_unicode": True,
        "reject_markup": True,
        "reject_urls": True,
        "remove_duplicates": True,
    },

    "filtering_is_quality_scoring": False,

    "source_record_id_definition": (
        "Original zero-based row index in the "
        "OpenSubtitles train split"
    ),
}

TASK_CONFIG["eligible_pool"] = POOL_INFO


# ------------------------------------------------------------
# 19. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("SOURCE POOL VALIDATION SUMMARY")
print("-" * 70)

print("✓ Clean eligible source pool created")
print("✓ Original OpenSubtitles row IDs preserved")
print("✓ Deterministic sampling used")
print("✓ Duplicate bilingual pairs removed")
print("✓ Invisible Unicode removed")
print("✓ Obvious subtitle markup removed")
print("✓ URLs removed")
print("✓ Source-copy contamination rejected")
print("✓ Excessive English on Chinese side rejected")
print("✓ Basic language and length checks passed")

print("\nIMPORTANT:")

print(
    "Cell 4 performs source/reference quality control only."
)

print(
    "It does NOT assign high/medium/low/very-low buckets."
)

print(
    "It does NOT create candidate translations."
)

print(
    "The next cell will select exactly 100 EN→ZH and "
    "100 ZH→EN base examples from this clean pool."
)




Source split: train
Total source records: 19,628,420

Pool construction configuration:
  Maximum records to inspect: 20,000
  Eligible pool target:       2,000
  Random seed:                42

Creating deterministic source indices...
✓ Generated 20,000 deterministic original row indices
✓ All sampled row indices are unique

Scanning and validating source records...

----------------------------------------------------------------------
POOL CONSTRUCTION RESULTS
----------------------------------------------------------------------
Records inspected: 2,927
Eligible records:  2,000
Acceptance rate:   68.3%

Rejected records:
  english_too_short                675
  chinese_english_contamination    167
  chinese_too_short                81
  empty                            2
  chinese_markup                   2

✓ Pool contains enough clean records for all 200 final Task A examples
✓ Eligible pool target reached: 2,000
✓ No duplicate bilingual pairs
✓ Source record IDs are unique
✓ Sou

In [14]:
#SELECT FINAL BASE EXAMPLES PER DIRECTION


import random
from collections import Counter




assert "eligible_pool" in globals(), (
    "eligible_pool does not exist. Run Cell 4 first."
)

assert len(eligible_pool) >= TOTAL_TARGET, (
    f"Need at least {TOTAL_TARGET} clean bilingual pairs, "
    f"but only {len(eligible_pool)} are available."
)

print(f"\nClean eligible pool: {len(eligible_pool):,}")
print(f"Target per direction: {TARGET_PER_DIRECTION}")
print(f"Total target: {TOTAL_TARGET}")


FINAL_SELECTION_SEED = SEED + 100

selection_rng = random.Random(
    FINAL_SELECTION_SEED
)

print(
    f"\nFinal selection seed: "
    f"{FINAL_SELECTION_SEED}"
)



selected_positions = selection_rng.sample(
    range(len(eligible_pool)),
    TOTAL_TARGET
)

selected_pairs = [
    eligible_pool[position]
    for position in selected_positions
]

assert len(selected_pairs) == TOTAL_TARGET

selected_record_ids = [
    row["source_record_id"]
    for row in selected_pairs
]

assert (
    len(selected_record_ids)
    == len(set(selected_record_ids))
), "Duplicate source records were selected."

print(
    f"✓ Selected {len(selected_pairs)} "
    "distinct clean bilingual pairs"
)


# ------------------------------------------------------------
# 4. ASSIGN DIRECTION
# ------------------------------------------------------------

en_zh_pairs = selected_pairs[
    :TARGET_PER_DIRECTION
]

zh_en_pairs = selected_pairs[
    TARGET_PER_DIRECTION:
]

assert len(en_zh_pairs) == TARGET_PER_DIRECTION
assert len(zh_en_pairs) == TARGET_PER_DIRECTION

print(
    f"✓ Assigned {len(en_zh_pairs)} pairs to EN -> ZH"
)

print(
    f"✓ Assigned {len(zh_en_pairs)} pairs to ZH -> EN"
)




base_examples = []


# ------------------------------------------------------------
# EN -> ZH
# ------------------------------------------------------------

for i, row in enumerate(
    en_zh_pairs,
    start=1
):

    example_id = f"TASKA-ENZH-{i:03d}"

    base_examples.append(
        {
            "example_id": example_id,

            "direction": "en-zh",

            "source_language": "English",

            "target_language": "Mandarin Chinese",

            # Directional text shown later to reviewer.
            "source_text": row["english"],

            "reference_translation": row["chinese"],

            # Preserve original corpus orientation.
            "original_source_text": row["english"],

            "original_reference_text": row["chinese"],

            # Traceability.
            "source_dataset": row["source_dataset"],

            "source_config": row["source_config"],

            "source_split": row["source_split"],

            "source_record_id": int(
                row["source_record_id"]
            ),

            "dataset_version": DATASET_VERSION,

            # Useful internal diagnostics.
            "english_token_count": int(
                row["english_token_count"]
            ),

            "chinese_char_count": int(
                row["chinese_char_count"]
            ),
        }
    )


# ------------------------------------------------------------
# ZH -> EN
# ------------------------------------------------------------

for i, row in enumerate(
    zh_en_pairs,
    start=1
):

    example_id = f"TASKA-ZHEN-{i:03d}"

    base_examples.append(
        {
            "example_id": example_id,

            "direction": "zh-en",

            "source_language": "Mandarin Chinese",

            "target_language": "English",

            # Reverse the bilingual pair for this direction.
            "source_text": row["chinese"],

            "reference_translation": row["english"],

            # Preserve ORIGINAL OpenSubtitles orientation.
            #
            # OpenSubtitles row:
            #     English -> Chinese
            #
            # even though this Task A item is:
            #     Chinese -> English
            "original_source_text": row["english"],

            "original_reference_text": row["chinese"],

            # Traceability.
            "source_dataset": row["source_dataset"],

            "source_config": row["source_config"],

            "source_split": row["source_split"],

            "source_record_id": int(
                row["source_record_id"]
            ),

            "dataset_version": DATASET_VERSION,

            "english_token_count": int(
                row["english_token_count"]
            ),

            "chinese_char_count": int(
                row["chinese_char_count"]
            ),
        }
    )


# ------------------------------------------------------------
# 6. VALIDATE EXACT COUNTS
# ------------------------------------------------------------

direction_counts = Counter(
    row["direction"]
    for row in base_examples
)

print("\nDirection counts:")

for direction in DIRECTIONS:

    print(
        f"  {direction}: "
        f"{direction_counts[direction]}"
    )

    assert (
        direction_counts[direction]
        == TARGET_PER_DIRECTION
    )


assert len(base_examples) == TOTAL_TARGET

print(
    f"\n✓ Exact total confirmed: "
    f"{len(base_examples)} examples"
)


# ------------------------------------------------------------
# 7. VALIDATE EXAMPLE IDs
# ------------------------------------------------------------

example_ids = [
    row["example_id"]
    for row in base_examples
]

assert (
    len(example_ids)
    == len(set(example_ids))
), "Duplicate example IDs detected."

print("✓ All Task A example IDs are unique")


# ------------------------------------------------------------
# 8. VALIDATE SOURCE RECORD UNIQUENESS
# ------------------------------------------------------------

final_source_record_ids = [
    row["source_record_id"]
    for row in base_examples
]

assert (
    len(final_source_record_ids)
    == len(set(final_source_record_ids))
), (
    "An underlying bilingual pair was reused "
    "across directions."
)

print(
    "✓ No underlying OpenSubtitles pair is "
    "reused across directions"
)


# ------------------------------------------------------------
# 9. VALIDATE SOURCE PROVENANCE
# ------------------------------------------------------------

for row in base_examples:

    assert row["source_dataset"] == DATASET_ID

    assert row["source_split"] == SOURCE_SPLIT

    assert isinstance(
        row["source_record_id"],
        int
    )

    assert (
        0
        <= row["source_record_id"]
        < len(source_data)
    )

print("✓ Source provenance fields validated")


# ------------------------------------------------------------
# 10. VALIDATE DIRECTION ORIENTATION
# ------------------------------------------------------------

for row in base_examples:

    if row["direction"] == "en-zh":

        assert (
            latin_letter_count(
                row["source_text"]
            )
            >= 4
        )

        assert contains_chinese(
            row["reference_translation"]
        )

    elif row["direction"] == "zh-en":

        assert contains_chinese(
            row["source_text"]
        )

        assert (
            latin_letter_count(
                row["reference_translation"]
            )
            >= 4
        )

    else:

        raise ValueError(
            f"Unexpected direction: "
            f"{row['direction']}"
        )


print("✓ Direction orientation checks passed")


# ------------------------------------------------------------
# 11. RE-CHECK CONTAMINATION
# ------------------------------------------------------------
# Cell 4 already filtered contamination.
# We verify the final selected subset again.

for row in base_examples:

    original_en = row["original_source_text"]
    original_zh = row["original_reference_text"]

    assert not source_copied_into_target(
        original_en,
        original_zh
    )

    assert latin_ratio(
        original_zh
    ) <= 0.30


print(
    "✓ Final selected records pass "
    "contamination checks"
)


# ------------------------------------------------------------
# 12. VERIFY INVISIBLE UNICODE IS GONE
# ------------------------------------------------------------

for row in base_examples:

    assert not has_format_character(
        row["source_text"]
    )

    assert not has_format_character(
        row["reference_translation"]
    )


print(
    "✓ Final selected text contains no "
    "Unicode formatting characters"
)


# ------------------------------------------------------------
# 13. SHOW EN -> ZH SAMPLE
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("EN -> ZH BASE SAMPLE")
print("-" * 70)

en_zh_sample = [
    row
    for row in base_examples
    if row["direction"] == "en-zh"
][:5]

for row in en_zh_sample:

    print(
        f"\n{row['example_id']}"
    )

    print(
        f"Source record ID: "
        f"{row['source_record_id']}"
    )

    print(
        f"Source:    "
        f"{row['source_text']}"
    )

    print(
        f"Reference: "
        f"{row['reference_translation']}"
    )


# ------------------------------------------------------------
# 14. SHOW ZH -> EN SAMPLE
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("ZH -> EN BASE SAMPLE")
print("-" * 70)

zh_en_sample = [
    row
    for row in base_examples
    if row["direction"] == "zh-en"
][:5]

for row in zh_en_sample:

    print(
        f"\n{row['example_id']}"
    )

    print(
        f"Source record ID: "
        f"{row['source_record_id']}"
    )

    print(
        f"Source:    "
        f"{row['source_text']}"
    )

    print(
        f"Reference: "
        f"{row['reference_translation']}"
    )


# ------------------------------------------------------------
# 15. SAVE SELECTION METADATA
# ------------------------------------------------------------

SELECTION_INFO = {
    "selection_seed": FINAL_SELECTION_SEED,

    "selection_method": (
        "deterministic_random_without_replacement"
    ),

    "eligible_pool_size": (
        len(eligible_pool)
    ),

    "selected_total": (
        len(base_examples)
    ),

    "target_per_direction": (
        TARGET_PER_DIRECTION
    ),

    "direction_counts": (
        dict(direction_counts)
    ),

    "unique_underlying_source_records": (
        len(set(final_source_record_ids))
    ),

    "underlying_pairs_reused_across_directions": False,

    "quality_buckets_assigned": False,

    "candidate_translations_created": False,
}

TASK_CONFIG[
    "final_base_selection"
] = SELECTION_INFO


# ------------------------------------------------------------
# 16. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("BASE SELECTION SUMMARY")
print("-" * 70)

print(
    f"✓ EN -> ZH: "
    f"{direction_counts['en-zh']}"
)

print(
    f"✓ ZH -> EN: "
    f"{direction_counts['zh-en']}"
)

print(
    f"✓ Total: "
    f"{len(base_examples)}"
)

print(
    "✓ 200 distinct underlying "
    "OpenSubtitles records"
)

print(
    "✓ Provenance retained for every example"
)

print(
    "✓ Cleaned text retained"
)

print(
    "✓ Stable Task A example IDs assigned"
)

print("\nIMPORTANT:")

print(
    "These are the FINAL BASE examples."
)

print(
    "No candidate translations have been "
    "constructed yet."
)

print(
    "No internal quality buckets have been "
    "assigned yet."
)

print(
    "Cell 6 will construct the controlled "
    "quality spectrum for human review."
)




Clean eligible pool: 2,000
Target per direction: 100
Total target: 200

Final selection seed: 142
✓ Selected 200 distinct clean bilingual pairs
✓ Assigned 100 pairs to EN -> ZH
✓ Assigned 100 pairs to ZH -> EN

Direction counts:
  en-zh: 100
  zh-en: 100

✓ Exact total confirmed: 200 examples
✓ All Task A example IDs are unique
✓ No underlying OpenSubtitles pair is reused across directions
✓ Source provenance fields validated
✓ Direction orientation checks passed
✓ Final selected records pass contamination checks
✓ Final selected text contains no Unicode formatting characters

----------------------------------------------------------------------
EN -> ZH BASE SAMPLE
----------------------------------------------------------------------

TASKA-ENZH-001
Source record ID: 14702640
Source:    - You wanted to see me, right?
Reference: 你有事找我 是的

TASKA-ENZH-002
Source record ID: 19242688
Source:    But to send these kids to a good school or let them do extracurricular activities...
Referenc

In [15]:
#  CONSTRUCT CONTROLLED QUALITY-SPECTRUM CANDIDATES

import copy
import random
import re
from collections import Counter


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

QUALITY_SEED = SEED + 200
quality_rng = random.Random(QUALITY_SEED)

QUALITY_BUCKETS = [
    "high",
    "medium",
    "low",
    "very_low",
]

QUALITY_PER_BUCKET_PER_DIRECTION = (
    TARGET_PER_DIRECTION // len(QUALITY_BUCKETS)
)

assert (
    QUALITY_PER_BUCKET_PER_DIRECTION
    * len(QUALITY_BUCKETS)
    == TARGET_PER_DIRECTION
)

print(f"\nQuality assignment seed: {QUALITY_SEED}")
print(
    "Examples per quality bucket per direction: "
    f"{QUALITY_PER_BUCKET_PER_DIRECTION}"
)


# ------------------------------------------------------------
# 2. INTERNAL BUCKET DEFINITIONS
# ------------------------------------------------------------

QUALITY_DEFINITIONS = {

    "high": (
        "No intentional degradation; reference translation "
        "is preserved."
    ),

    "medium": (
        "Mostly correct translation with a small observable "
        "surface, fluency, or low-content omission."
    ),

    "low": (
        "Substantial partial omission that removes meaningful "
        "translation content while preserving some meaning."
    ),

    "very_low": (
        "Major failure such as severe omission or "
        "wrong-language/source-copy output."
    ),
}

print("\nInternal construction buckets:")

for bucket in QUALITY_BUCKETS:
    print(
        f"  {bucket:<10} "
        f"{QUALITY_DEFINITIONS[bucket]}"
    )


# ------------------------------------------------------------
# 3. ENGLISH TOKEN HELPERS
# ------------------------------------------------------------

def english_words(text):
    """
    Return word spans from English text.
    """

    return list(
        re.finditer(
            r"[A-Za-z0-9]+(?:['’-][A-Za-z0-9]+)*",
            text
        )
    )


def remove_english_word_span(text, start_word, end_word):
    """
    Remove English words using word indices [start_word:end_word].
    """

    matches = english_words(text)

    if not matches:
        return text

    start_word = max(
        0,
        min(start_word, len(matches) - 1)
    )

    end_word = max(
        start_word + 1,
        min(end_word, len(matches))
    )

    start_char = matches[start_word].start()
    end_char = matches[end_word - 1].end()

    candidate = (
        text[:start_char]
        + text[end_char:]
    )

    candidate = normalize_whitespace(candidate)

    # Clean spacing before punctuation.
    candidate = re.sub(
        r"\s+([,.!?;:])",
        r"\1",
        candidate
    )

    return candidate.strip()


# ------------------------------------------------------------
# 4. CHINESE CHARACTER HELPERS
# ------------------------------------------------------------

def chinese_positions(text):

    return [
        i
        for i, char in enumerate(text)
        if re.match(
            r"[\u3400-\u4DBF\u4E00-\u9FFF]",
            char
        )
    ]


def remove_chinese_span_by_fraction(
    text,
    fraction,
):
    """
    Remove a contiguous middle span containing approximately
    `fraction` of the Chinese characters.
    """

    positions = chinese_positions(text)

    if len(positions) < 4:
        return text

    remove_count = max(
        1,
        int(round(len(positions) * fraction))
    )

    # Never remove every Chinese character.
    remove_count = min(
        remove_count,
        len(positions) - 2
    )

    start_idx = max(
        0,
        (len(positions) - remove_count) // 2
    )

    end_idx = (
        start_idx
        + remove_count
        - 1
    )

    start_char = positions[start_idx]
    end_char = positions[end_idx] + 1

    candidate = (
        text[:start_char]
        + text[end_char:]
    )

    return normalize_whitespace(candidate)


# ------------------------------------------------------------
# 5. MEDIUM ENGLISH
# ------------------------------------------------------------
# Medium should be visibly different but still largely correct.

def medium_english(reference):

    candidate = reference.strip()

    # A. Remove comma.
    if "," in candidate:

        modified = candidate.replace(
            ",",
            "",
            1
        )

        modified = clean_text(modified)

        if modified != clean_text(reference):
            return (
                modified,
                "minor_punctuation_omission"
            )

    # B. Remove terminal punctuation.
    if candidate.endswith(
        (".", "!", "?", "…")
    ):

        modified = candidate.rstrip(
            ".!?…"
        ).strip()

        modified = clean_text(modified)

        if modified != clean_text(reference):
            return (
                modified,
                "minor_terminal_punctuation_omission"
            )

    # C. Remove a low-content discourse/function word.
    removable_words = [
        "really",
        "just",
        "actually",
        "please",
        "well",
        "still",
        "already",
        "very",
    ]

    for word in removable_words:

        pattern = rf"\b{re.escape(word)}\b"

        if re.search(
            pattern,
            candidate,
            flags=re.IGNORECASE
        ):

            modified = re.sub(
                pattern,
                "",
                candidate,
                count=1,
                flags=re.IGNORECASE
            )

            modified = clean_text(modified)

            if modified != clean_text(reference):
                return (
                    modified,
                    "minor_low_content_word_omission"
                )

    # D. Guaranteed observable fallback:
    # remove the first punctuation mark if available.
    punctuation_match = re.search(
        r"[,.;:!?]",
        candidate
    )

    if punctuation_match:

        i = punctuation_match.start()

        modified = (
            candidate[:i]
            + candidate[i + 1:]
        )

        modified = clean_text(modified)

        if modified != clean_text(reference):
            return (
                modified,
                "minor_punctuation_omission"
            )

    # E. Final fallback:
    # remove one short function word if possible.
    matches = english_words(candidate)

    function_words = {
        "a", "an", "the", "to", "of",
        "in", "on", "at", "for",
    }

    for i, match in enumerate(matches):

        if match.group().casefold() in function_words:

            modified = remove_english_word_span(
                candidate,
                i,
                i + 1
            )

            if (
                modified
                and
                clean_text(modified)
                != clean_text(reference)
            ):
                return (
                    modified,
                    "minor_function_word_omission"
                )

    # F. Last guaranteed fallback:
    # remove the final word.
    if len(matches) >= 4:

        modified = remove_english_word_span(
            candidate,
            len(matches) - 1,
            len(matches)
        )

        return (
            modified,
            "minor_final_word_omission"
        )

    # For unusually short records.
    modified = candidate + "."

    if clean_text(modified) == clean_text(reference):
        modified = candidate + "!"

    return (
        modified,
        "minor_surface_variation"
    )


# ------------------------------------------------------------
# 6. MEDIUM CHINESE
# ------------------------------------------------------------

def medium_chinese(reference):

    candidate = reference.strip()

    # A. Remove punctuation.
    punctuation = [
        "，", "。", "！", "？",
        ",", ".", "!", "?",
        "、", "；", ";",
    ]

    for mark in punctuation:

        if mark in candidate:

            modified = candidate.replace(
                mark,
                "",
                1
            )

            modified = clean_text(modified)

            if modified != clean_text(reference):
                return (
                    modified,
                    "minor_punctuation_omission"
                )

    # B. Remove discourse particle.
    particles = [
        "啊", "呀", "吧", "呢",
        "嘛", "哦", "啦",
    ]

    for particle in particles:

        if particle in candidate:

            modified = candidate.replace(
                particle,
                "",
                1
            )

            modified = clean_text(modified)

            if modified != clean_text(reference):
                return (
                    modified,
                    "minor_discourse_particle_omission"
                )

    # C. Remove a low-content Chinese function character.
    low_content = [
        "的", "了", "着", "过",
    ]

    for char in low_content:

        if char in candidate:

            modified = candidate.replace(
                char,
                "",
                1
            )

            modified = clean_text(modified)

            if modified != clean_text(reference):
                return (
                    modified,
                    "minor_function_character_omission"
                )

    # D. Guaranteed fallback:
    # remove one Chinese character near the end.
    positions = chinese_positions(candidate)

    if len(positions) >= 5:

        target = positions[-2]

        modified = (
            candidate[:target]
            + candidate[target + 1:]
        )

        modified = clean_text(modified)

        return (
            modified,
            "minor_single_character_omission"
        )

    # Extremely short fallback.
    modified = candidate + "。"

    if clean_text(modified) == clean_text(reference):
        modified = candidate + "！"

    return (
        modified,
        "minor_surface_variation"
    )


# ------------------------------------------------------------
# 7. LOW ENGLISH
# ------------------------------------------------------------
# Remove approximately 30% of English words from a contiguous
# middle region. This produces a substantial omission rather
# than arbitrary single-word corruption.

def low_english(reference):

    matches = english_words(reference)

    n_words = len(matches)

    if n_words >= 5:

        remove_count = max(
            2,
            int(round(n_words * 0.30))
        )

        remove_count = min(
            remove_count,
            n_words - 2
        )

        start = max(
            1,
            (n_words - remove_count) // 2
        )

        end = start + remove_count

        candidate = remove_english_word_span(
            reference,
            start,
            end
        )

        if (
            candidate
            and clean_text(candidate)
            != clean_text(reference)
        ):

            return (
                candidate,
                "substantial_contiguous_content_omission"
            )

    # Short-sentence fallback:
    # remove one content-bearing word.
    if n_words >= 3:

        target = n_words // 2

        candidate = remove_english_word_span(
            reference,
            target,
            target + 1
        )

        if candidate:
            return (
                candidate,
                "meaningful_content_word_omission"
            )

    return (
        reference[:max(1, len(reference) // 2)],
        "substantial_partial_omission"
    )


# ------------------------------------------------------------
# 8. LOW CHINESE
# ------------------------------------------------------------

def low_chinese(reference):

    candidate = remove_chinese_span_by_fraction(
        reference,
        0.30
    )

    if (
        candidate
        and clean_text(candidate)
        != clean_text(reference)
    ):

        return (
            candidate,
            "substantial_contiguous_content_omission"
        )

    positions = chinese_positions(reference)

    if len(positions) >= 3:

        target = positions[
            len(positions) // 2
        ]

        candidate = (
            reference[:target]
            + reference[target + 1:]
        )

        return (
            clean_text(candidate),
            "meaningful_content_character_omission"
        )

    return (
        reference[:max(1, len(reference) // 2)],
        "substantial_partial_omission"
    )


# ------------------------------------------------------------
# 9. VERY LOW
# ------------------------------------------------------------
# Alternate between:
#
#   A. source-copy / wrong-language output
#   B. severe translation omission
#
# Both are realistic judge failure modes and clearly distinct
# from the LOW category.

def very_low_candidate(row, bucket_index):

    source = row["source_text"]
    reference = row["reference_translation"]
    direction = row["direction"]


    # --------------------------------------------------------
    # A. WRONG-LANGUAGE SOURCE COPY
    # --------------------------------------------------------

    if bucket_index % 2 == 0:

        return (
            source,
            "wrong_language_source_copy"
        )


    # --------------------------------------------------------
    # B. SEVERE OMISSION
    # --------------------------------------------------------

    if direction == "en-zh":

        positions = chinese_positions(
            reference
        )

        if len(positions) >= 4:

            keep_count = max(
                2,
                int(round(
                    len(positions) * 0.35
                ))
            )

            keep_count = min(
                keep_count,
                len(positions) - 1
            )

            end_char = (
                positions[keep_count - 1]
                + 1
            )

            candidate = reference[
                :end_char
            ]

            candidate = clean_text(
                candidate
            )

            if candidate:
                return (
                    candidate,
                    "severe_translation_omission"
                )


    else:

        matches = english_words(
            reference
        )

        n_words = len(matches)

        if n_words >= 3:

            keep_count = max(
                2,
                int(round(
                    n_words * 0.35
                ))
            )

            keep_count = min(
                keep_count,
                n_words - 1
            )

            end_char = (
                matches[keep_count - 1].end()
            )

            candidate = reference[
                :end_char
            ]

            candidate = clean_text(
                candidate
            )

            if candidate:
                return (
                    candidate,
                    "severe_translation_omission"
                )


    # Guaranteed fallback.
    return (
        source,
        "wrong_language_source_copy"
    )


# ------------------------------------------------------------
# 10. ASSIGN QUALITY BUCKETS
# ------------------------------------------------------------

quality_assignment = {}

for direction in DIRECTIONS:

    direction_ids = [
        row["example_id"]
        for row in base_examples
        if row["direction"] == direction
    ]

    assert (
        len(direction_ids)
        == TARGET_PER_DIRECTION
    )

    quality_rng.shuffle(
        direction_ids
    )

    cursor = 0

    for bucket in QUALITY_BUCKETS:

        next_cursor = (
            cursor
            + QUALITY_PER_BUCKET_PER_DIRECTION
        )

        bucket_ids = direction_ids[
            cursor:next_cursor
        ]

        for example_id in bucket_ids:

            quality_assignment[
                example_id
            ] = bucket

        cursor = next_cursor


assert len(quality_assignment) == TOTAL_TARGET

print(
    "\n✓ Internal quality buckets assigned "
    "independently within each direction"
)


# ------------------------------------------------------------
# 11. CREATE CANDIDATES
# ------------------------------------------------------------

review_candidates = []

very_low_indices = {
    "en-zh": 0,
    "zh-en": 0,
}


for row in base_examples:

    item = copy.deepcopy(row)

    bucket = quality_assignment[
        row["example_id"]
    ]

    reference = clean_text(
        row["reference_translation"]
    )


    if bucket == "high":

        candidate = reference

        method = (
            "reference_preserved_no_intentional_error"
        )


    elif bucket == "medium":

        if row["direction"] == "en-zh":

            candidate, method = medium_chinese(
                reference
            )

        else:

            candidate, method = medium_english(
                reference
            )


    elif bucket == "low":

        if row["direction"] == "en-zh":

            candidate, method = low_chinese(
                reference
            )

        else:

            candidate, method = low_english(
                reference
            )


    elif bucket == "very_low":

        index = very_low_indices[
            row["direction"]
        ]

        candidate, method = very_low_candidate(
            row,
            index
        )

        very_low_indices[
            row["direction"]
        ] += 1


    else:

        raise ValueError(
            f"Unknown quality bucket: {bucket}"
        )


    candidate = clean_text(candidate)


    # --------------------------------------------------------
    # GUARANTEE NON-HIGH CANDIDATES DIFFER
    # --------------------------------------------------------

    if (
        bucket != "high"
        and candidate == reference
    ):

        if row["direction"] == "en-zh":

            positions = chinese_positions(
                reference
            )

            if len(positions) >= 2:

                target = positions[-1]

                candidate = (
                    reference[:target]
                    + reference[target + 1:]
                )

                candidate = clean_text(
                    candidate
                )

                method += (
                    "+forced_final_character_omission"
                )

        else:

            matches = english_words(
                reference
            )

            if len(matches) >= 2:

                candidate = remove_english_word_span(
                    reference,
                    len(matches) - 1,
                    len(matches)
                )

                candidate = clean_text(
                    candidate
                )

                method += (
                    "+forced_final_word_omission"
                )


    assert candidate, (
        f"Empty candidate generated for "
        f"{row['example_id']}"
    )


    # Critical invariant.
    if bucket == "high":

        assert candidate == reference

    else:

        assert candidate != reference, (
            f"Non-high candidate remained identical: "
            f"{row['example_id']}"
        )


    item[
        "candidate_translation"
    ] = candidate

    item[
        "internal_quality_bucket"
    ] = bucket

    item[
        "candidate_generation_method"
    ] = method

    review_candidates.append(
        item
    )


# ------------------------------------------------------------
# 12. DISTRIBUTION VALIDATION
# ------------------------------------------------------------

quality_counts = Counter(
    (
        row["direction"],
        row["internal_quality_bucket"]
    )
    for row in review_candidates
)

print("\n" + "-" * 70)
print("QUALITY DISTRIBUTION")
print("-" * 70)


for direction in DIRECTIONS:

    print(f"\n{direction}:")

    for bucket in QUALITY_BUCKETS:

        count = quality_counts[
            (direction, bucket)
        ]

        print(
            f"  {bucket:<10} "
            f"{count}"
        )

        assert (
            count
            == QUALITY_PER_BUCKET_PER_DIRECTION
        )


assert len(review_candidates) == TOTAL_TARGET

print(
    f"\n✓ Total candidate rows: "
    f"{len(review_candidates)}"
)


# ------------------------------------------------------------
# 13. HIGH / NON-HIGH VALIDATION
# ------------------------------------------------------------

high_rows = [
    row
    for row in review_candidates
    if row["internal_quality_bucket"] == "high"
]

non_high_rows = [
    row
    for row in review_candidates
    if row["internal_quality_bucket"] != "high"
]


assert all(
    row["candidate_translation"]
    == row["reference_translation"]
    for row in high_rows
)

assert all(
    row["candidate_translation"]
    != row["reference_translation"]
    for row in non_high_rows
)


print(
    "✓ All high candidates exactly preserve "
    "their references"
)

print(
    "✓ All 150 non-high candidates differ "
    "from their references"
)


# ------------------------------------------------------------
# 14. LANGUAGE FAILURE VALIDATION
# ------------------------------------------------------------

wrong_language_rows = [
    row
    for row in review_candidates
    if row["candidate_generation_method"]
    == "wrong_language_source_copy"
]

print(
    f"✓ Wrong-language/source-copy cases: "
    f"{len(wrong_language_rows)}"
)


# ------------------------------------------------------------
# 15. METHOD DISTRIBUTION
# ------------------------------------------------------------

method_counts = Counter(
    row["candidate_generation_method"]
    for row in review_candidates
)

print("\n" + "-" * 70)
print("CANDIDATE GENERATION METHODS")
print("-" * 70)

for method, count in method_counts.most_common():

    print(
        f"  {method:<48} "
        f"{count}"
    )


# ------------------------------------------------------------
# 16. DISPLAY TWO EXAMPLES PER BUCKET
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("QUALITY-SPECTRUM EXAMPLES")
print("-" * 70)


for direction in DIRECTIONS:

    print(
        f"\n{'=' * 24} "
        f"{direction.upper()} "
        f"{'=' * 24}"
    )

    for bucket in QUALITY_BUCKETS:

        rows = [
            row
            for row in review_candidates
            if (
                row["direction"] == direction
                and
                row["internal_quality_bucket"] == bucket
            )
        ]

        print(
            f"\n[{bucket.upper()}]"
        )

        for sample in rows[:2]:

            print(
                f"\nID:        "
                f"{sample['example_id']}"
            )

            print(
                f"Source:    "
                f"{sample['source_text']}"
            )

            print(
                f"Reference: "
                f"{sample['reference_translation']}"
            )

            print(
                f"Candidate: "
                f"{sample['candidate_translation']}"
            )

            print(
                f"Method:    "
                f"{sample['candidate_generation_method']}"
            )


# ------------------------------------------------------------
# 17. SAVE METADATA
# ------------------------------------------------------------

QUALITY_CONSTRUCTION_INFO = {

    "quality_seed": QUALITY_SEED,

    "buckets": QUALITY_BUCKETS,

    "bucket_definitions": QUALITY_DEFINITIONS,

    "examples_per_bucket_per_direction": (
        QUALITY_PER_BUCKET_PER_DIRECTION
    ),

    "quality_counts": {
        f"{direction}:{bucket}": (
            quality_counts[
                (direction, bucket)
            ]
        )
        for direction in DIRECTIONS
        for bucket in QUALITY_BUCKETS
    },

    "internal_buckets_are_human_labels": False,

    "reviewer_should_see_internal_bucket": False,

    "reviewer_should_see_generation_method": False,

    "candidate_generation_methods": (
        dict(method_counts)
    ),

    "all_non_high_candidates_differ_from_reference": True,
}

TASK_CONFIG[
    "quality_construction"
] = QUALITY_CONSTRUCTION_INFO


# ------------------------------------------------------------
# 18. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("QUALITY CONSTRUCTION SUMMARY")
print("-" * 70)

print("✓ 200 candidate translations created")
print("✓ 100 EN -> ZH")
print("✓ 100 ZH -> EN")

print(
    "✓ 25 high / 25 medium / 25 low / "
    "25 very-low per direction"
)

print(
    "✓ High candidates preserve references"
)

print(
    "✓ Every non-high candidate differs "
    "after final cleaning"
)

print(
    "✓ Low candidates use substantial "
    "partial omissions"
)

print(
    "✓ Very-low candidates use severe omission "
    "or wrong-language/source-copy failures"
)

print(
    "✓ Internal construction metadata retained"
)

print("\nIMPORTANT:")

print(
    "The quality bucket remains an INTERNAL "
    "construction variable."
)

print(
    "It is not a human score and will not be "
    "shown to reviewers."
)

print(
    "Human reviewers will independently score "
    "the candidate translations using the rubric."
)



Quality assignment seed: 242
Examples per quality bucket per direction: 25

Internal construction buckets:
  high       No intentional degradation; reference translation is preserved.
  medium     Mostly correct translation with a small observable surface, fluency, or low-content omission.
  low        Substantial partial omission that removes meaningful translation content while preserving some meaning.
  very_low   Major failure such as severe omission or wrong-language/source-copy output.

✓ Internal quality buckets assigned independently within each direction

----------------------------------------------------------------------
QUALITY DISTRIBUTION
----------------------------------------------------------------------

en-zh:
  high       25
  medium     25
  low        25
  very_low   25

zh-en:
  high       25
  medium     25
  low        25
  very_low   25

✓ Total candidate rows: 200
✓ All high candidates exactly preserve their references
✓ All 150 non-high candidates differ

In [16]:
# = BLIND + SHUFFLE HUMAN-REVIEW DATASET



import copy
import random
from collections import Counter


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

REVIEW_SHUFFLE_SEED = SEED + 300

review_rng = random.Random(
    REVIEW_SHUFFLE_SEED
)

print(
    f"\nReviewer shuffle seed: "
    f"{REVIEW_SHUFFLE_SEED}"
)


# ------------------------------------------------------------
# 2. VALIDATE INPUT
# ------------------------------------------------------------

assert "review_candidates" in globals(), (
    "review_candidates does not exist. "
    "Run Cell 6 first."
)

assert len(review_candidates) == TOTAL_TARGET

print(
    f"Input candidate rows: "
    f"{len(review_candidates)}"
)




REVIEWER_FIELDS = [
    "review_id",
    "example_id",
    "direction",
    "source_text",
    "candidate_translation",
    "reference_translation",
    "human_score",
    "reviewer_comment",
]



INTERNAL_FIELDS = [
    "review_id",
    "example_id",
    "direction",
    "source_language",
    "target_language",
    "source_dataset",
    "source_config",
    "source_split",
    "source_record_id",
    "original_source_text",
    "original_reference_text",
    "reference_translation",
    "candidate_translation",
    "internal_quality_bucket",
    "candidate_generation_method",
    "dataset_version",
]


print("\nReviewer-visible fields:")

for field in REVIEWER_FIELDS:
    print(f"  ✓ {field}")


print("\nInternal-only construction/provenance fields:")

for field in [
    "source_dataset",
    "source_config",
    "source_split",
    "source_record_id",
    "internal_quality_bucket",
    "candidate_generation_method",
    "dataset_version",
]:
    print(f"  - {field}")




shuffled_indices = list(
    range(len(review_candidates))
)

review_rng.shuffle(
    shuffled_indices
)

assert (
    len(shuffled_indices)
    == TOTAL_TARGET
)

assert (
    len(set(shuffled_indices))
    == TOTAL_TARGET
)

print(
    "\n✓ Candidate order shuffled "
    "deterministically"
)



reviewer_dataset = []
internal_metadata = []


for review_position, candidate_index in enumerate(
    shuffled_indices,
    start=1
):

    row = review_candidates[
        candidate_index
    ]

    # --------------------------------------------------------
    # Anonymous review ID
    # --------------------------------------------------------

    review_id = (
        f"MR-{review_position:03d}"
    )


    reviewer_row = {
        "review_id": review_id,

        "example_id": row[
            "example_id"
        ],

        "direction": row[
            "direction"
        ],

        "source_text": row[
            "source_text"
        ],

        "candidate_translation": row[
            "candidate_translation"
        ],

        "reference_translation": row[
            "reference_translation"
        ],

        # Blank fields for human annotation.
        "human_score": "",

        "reviewer_comment": "",
    }

    reviewer_dataset.append(
        reviewer_row
    )


    # --------------------------------------------------------
    # INTERNAL TRACEABILITY RECORD
    # --------------------------------------------------------

    internal_row = {
        "review_id": review_id,

        "example_id": row[
            "example_id"
        ],

        "direction": row[
            "direction"
        ],

        "source_language": row[
            "source_language"
        ],

        "target_language": row[
            "target_language"
        ],

        "source_dataset": row[
            "source_dataset"
        ],

        "source_config": row[
            "source_config"
        ],

        "source_split": row[
            "source_split"
        ],

        "source_record_id": int(
            row["source_record_id"]
        ),

        "original_source_text": row[
            "original_source_text"
        ],

        "original_reference_text": row[
            "original_reference_text"
        ],

        "reference_translation": row[
            "reference_translation"
        ],

        "candidate_translation": row[
            "candidate_translation"
        ],

        "internal_quality_bucket": row[
            "internal_quality_bucket"
        ],

        "candidate_generation_method": row[
            "candidate_generation_method"
        ],

        "dataset_version": row[
            "dataset_version"
        ],
    }

    internal_metadata.append(
        internal_row
    )


# ------------------------------------------------------------
# 7. BASIC COUNT VALIDATION
# ------------------------------------------------------------

assert (
    len(reviewer_dataset)
    == TOTAL_TARGET
)

assert (
    len(internal_metadata)
    == TOTAL_TARGET
)

print(
    f"\n✓ Reviewer rows created: "
    f"{len(reviewer_dataset)}"
)

print(
    f"✓ Internal metadata rows created: "
    f"{len(internal_metadata)}"
)


# ------------------------------------------------------------
# 8. REVIEW-ID VALIDATION
# ------------------------------------------------------------

review_ids = [
    row["review_id"]
    for row in reviewer_dataset
]

assert (
    len(review_ids)
    == len(set(review_ids))
)

print("✓ All review IDs are unique")


# ------------------------------------------------------------
# 9. DIRECTION COUNTS
# ------------------------------------------------------------

review_direction_counts = Counter(
    row["direction"]
    for row in reviewer_dataset
)

print("\nReviewer dataset direction counts:")

for direction in DIRECTIONS:

    count = review_direction_counts[
        direction
    ]

    print(
        f"  {direction}: {count}"
    )

    assert (
        count
        == TARGET_PER_DIRECTION
    )


# ------------------------------------------------------------
# 10. VERIFY BLINDING
# ------------------------------------------------------------

FORBIDDEN_REVIEWER_FIELDS = {
    "internal_quality_bucket",
    "candidate_generation_method",
    "source_dataset",
    "source_config",
    "source_split",
    "source_record_id",
    "dataset_version",
    "quality_seed",
}


for row in reviewer_dataset:

    leaked_fields = (
        FORBIDDEN_REVIEWER_FIELDS
        .intersection(row.keys())
    )

    assert not leaked_fields, (
        f"Internal metadata leaked into "
        f"reviewer row: {leaked_fields}"
    )


print(
    "✓ Internal quality/construction metadata "
    "is hidden from reviewers"
)


# ------------------------------------------------------------
# 11. VERIFY HUMAN FIELDS ARE EMPTY
# ------------------------------------------------------------

assert all(
    row["human_score"] == ""
    for row in reviewer_dataset
)

assert all(
    row["reviewer_comment"] == ""
    for row in reviewer_dataset
)

print(
    "✓ Human annotation fields initialized blank"
)


# ------------------------------------------------------------
# 12. VERIFY REVIEWER/INTERNAL MAPPING
# ------------------------------------------------------------

internal_by_review_id = {
    row["review_id"]: row
    for row in internal_metadata
}


for reviewer_row in reviewer_dataset:

    review_id = reviewer_row[
        "review_id"
    ]

    assert (
        review_id
        in internal_by_review_id
    )

    internal_row = internal_by_review_id[
        review_id
    ]

    assert (
        reviewer_row["example_id"]
        == internal_row["example_id"]
    )

    assert (
        reviewer_row["direction"]
        == internal_row["direction"]
    )

    assert (
        reviewer_row["source_text"]
        == (
            internal_row["original_source_text"]
            if reviewer_row["direction"] == "en-zh"
            else internal_row["original_reference_text"]
        )
    )

    assert (
        reviewer_row[
            "candidate_translation"
        ]
        == internal_row[
            "candidate_translation"
        ]
    )

    assert (
        reviewer_row[
            "reference_translation"
        ]
        == internal_row[
            "reference_translation"
        ]
    )


print(
    "✓ Reviewer rows map correctly to "
    "internal traceability records"
)


# ------------------------------------------------------------
# 13. VERIFY QUALITY MIX AFTER SHUFFLING
# ------------------------------------------------------------
# This information is printed for us during construction,
# but it will NOT appear in the reviewer export.

internal_quality_counts = Counter(
    (
        row["direction"],
        row["internal_quality_bucket"]
    )
    for row in internal_metadata
)

print("\nInternal post-shuffle quality check:")

for direction in DIRECTIONS:

    print(f"\n  {direction}")

    for bucket in QUALITY_BUCKETS:

        count = internal_quality_counts[
            (direction, bucket)
        ]

        print(
            f"    {bucket:<10} "
            f"{count}"
        )

        assert (
            count
            == QUALITY_PER_BUCKET_PER_DIRECTION
        )


# ------------------------------------------------------------
# 14. CHECK THAT ORDER IS ACTUALLY MIXED
# ------------------------------------------------------------

first_20_internal_buckets = [
    internal_metadata[i][
        "internal_quality_bucket"
    ]
    for i in range(
        min(20, len(internal_metadata))
    )
]

print(
    "\nFirst 20 internal buckets "
    "(diagnostic only):"
)

print(
    "  "
    + ", ".join(
        first_20_internal_buckets
    )
)

assert len(
    set(first_20_internal_buckets)
) > 1, (
    "Review ordering does not appear mixed."
)

print("✓ Review order contains mixed quality levels")


# ------------------------------------------------------------
# 15. DISPLAY FIRST 5 REVIEWER ROWS
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("FIRST 5 BLINDED REVIEWER ROWS")
print("-" * 70)


for row in reviewer_dataset[:5]:

    print(
        f"\nReview ID: {row['review_id']}"
    )

    print(
        f"Example ID: {row['example_id']}"
    )

    print(
        f"Direction: {row['direction']}"
    )

    print(
        f"Source: {row['source_text']}"
    )

    print(
        "Candidate: "
        f"{row['candidate_translation']}"
    )

    print(
        "Reference: "
        f"{row['reference_translation']}"
    )

    print(
        "Human score: "
        f"{repr(row['human_score'])}"
    )

    print(
        "Reviewer comment: "
        f"{repr(row['reviewer_comment'])}"
    )


# ------------------------------------------------------------
# 16. SAVE REVIEW PACKAGING METADATA
# ------------------------------------------------------------

REVIEW_PACKAGING_INFO = {
    "shuffle_seed": REVIEW_SHUFFLE_SEED,

    "shuffle_method": (
        "deterministic_random_shuffle"
    ),

    "reviewer_rows": (
        len(reviewer_dataset)
    ),

    "internal_rows": (
        len(internal_metadata)
    ),

    "reviewer_fields": (
        REVIEWER_FIELDS
    ),

    "internal_fields": (
        INTERNAL_FIELDS
    ),

    "direction_counts": (
        dict(review_direction_counts)
    ),

    "internal_quality_hidden": True,

    "generation_method_hidden": True,

    "provenance_hidden_from_reviewer": True,

    "human_score_initialized_blank": True,

    "reviewer_comment_initialized_blank": True,
}

TASK_CONFIG[
    "review_packaging"
] = REVIEW_PACKAGING_INFO


# ------------------------------------------------------------
# 17. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("REVIEW PACKAGING SUMMARY")
print("-" * 70)

print(
    "✓ 200 examples prepared for human review"
)

print(
    "✓ Reviewer order shuffled deterministically"
)

print(
    "✓ Internal construction buckets hidden"
)

print(
    "✓ Candidate-generation methods hidden"
)

print(
    "✓ Source provenance retained separately"
)

print(
    "✓ Human score field left blank"
)

print(
    "✓ Reviewer comment field left blank"
)

print(
    "✓ Review/internal mapping validated"
)

print("\nIMPORTANT:")

print(
    "reviewer_dataset is the BLINDED dataset "
    "that will eventually be sent to reviewers."
)

print(
    "internal_metadata must remain internal and "
    "must NOT be sent to reviewers."
)

print(
    "The next cell will define the scoring rubric "
    "and reviewer instructions."
)




Reviewer shuffle seed: 342
Input candidate rows: 200

Reviewer-visible fields:
  ✓ review_id
  ✓ example_id
  ✓ direction
  ✓ source_text
  ✓ candidate_translation
  ✓ reference_translation
  ✓ human_score
  ✓ reviewer_comment

Internal-only construction/provenance fields:
  - source_dataset
  - source_config
  - source_split
  - source_record_id
  - internal_quality_bucket
  - candidate_generation_method
  - dataset_version

✓ Candidate order shuffled deterministically

✓ Reviewer rows created: 200
✓ Internal metadata rows created: 200
✓ All review IDs are unique

Reviewer dataset direction counts:
  en-zh: 100
  zh-en: 100
✓ Internal quality/construction metadata is hidden from reviewers
✓ Human annotation fields initialized blank
✓ Reviewer rows map correctly to internal traceability records

Internal post-shuffle quality check:

  en-zh
    high       25
    medium     25
    low        25
    very_low   25

  zh-en
    high       25
    medium     25
    low        25
    very_lo

In [17]:

# CELL 8: DEFINE HUMAN-REVIEW RUBRIC + INSTRUCTIONS


import json




SCORE_MIN = 0
SCORE_MAX = 4

ALLOWED_SCORES = list(
    range(SCORE_MIN, SCORE_MAX + 1)
)


SCORING_RUBRIC = {

    4: {
        "label": "Excellent",
        "definition": (
            "The candidate accurately preserves the source "
            "meaning and is natural and understandable in the "
            "target language. No meaningful translation error "
            "is present. Very small stylistic or punctuation "
            "differences that do not affect quality are allowed."
        ),
    },

    3: {
        "label": "Good",
        "definition": (
            "The candidate preserves the main source meaning "
            "and is understandable, but contains a minor error "
            "or awkwardness. The issue does not materially "
            "change the central meaning."
        ),
    },

    2: {
        "label": "Fair",
        "definition": (
            "The candidate communicates part or most of the "
            "source meaning, but contains a noticeable error, "
            "omission, addition, mistranslation, or fluency "
            "problem. Important information may be affected, "
            "but the translation remains partially useful."
        ),
    },

    1: {
        "label": "Poor",
        "definition": (
            "The candidate preserves only a limited portion of "
            "the source meaning or contains major translation "
            "errors. Important information is missing, wrong, "
            "or seriously distorted."
        ),
    },

    0: {
        "label": "Failed",
        "definition": (
            "The candidate is unusable as a translation of the "
            "source. Examples include wrong-language output, "
            "copying the source instead of translating it, "
            "unrelated output, nearly complete loss of meaning, "
            "or severe nonsense."
        ),
    },
}


# ------------------------------------------------------------
# 2. EVALUATION DIMENSIONS
# ------------------------------------------------------------
#
# These dimensions guide the reviewer toward one OVERALL
# integer score. They are not separately scored.

EVALUATION_DIMENSIONS = {

    "meaning_preservation": (
        "Does the candidate preserve the meaning, facts, "
        "relationships, intent, and important details of the "
        "source?"
    ),

    "completeness": (
        "Does the candidate translate the important source "
        "content without substantial omissions or unsupported "
        "additions?"
    ),

    "target_language_quality": (
        "Is the candidate understandable and reasonably "
        "natural in the target language?"
    ),

    "critical_errors": (
        "Does the candidate introduce negation errors, wrong "
        "entities, wrong quantities, wrong relationships, "
        "contradictions, or other changes that materially alter "
        "the source meaning?"
    ),
}


# ------------------------------------------------------------
# 3. REVIEWER DECISION RULES
# ------------------------------------------------------------

DECISION_RULES = [

    (
        "Judge the candidate primarily against the source text. "
        "Do not score based only on similarity to the reference."
    ),

    (
        "Use the reference translation as supporting context. "
        "The reference may contain awkward wording, omissions, "
        "or other imperfections."
    ),

    (
        "A candidate may deserve a score of 4 even when its "
        "wording differs from the reference, provided it "
        "faithfully translates the source."
    ),

    (
        "Do not penalize harmless differences in wording, "
        "punctuation, capitalization, or style unless they "
        "reduce meaning, clarity, or naturalness."
    ),

    (
        "Penalize omissions or additions according to how much "
        "important source meaning they affect."
    ),

    (
        "Meaning-changing errors such as incorrect negation, "
        "entities, quantities, actions, or relationships should "
        "receive a substantial penalty."
    ),

    (
        "If the candidate simply copies the source and therefore "
        "fails to translate it into the required target language, "
        "assign score 0."
    ),

    (
        "If the output is primarily in the wrong target language "
        "and is not a valid translation, assign score 0."
    ),

    (
        "Use exactly one integer score from 0 through 4. "
        "Do not use decimal or half-point scores."
    ),

    (
        "If uncertain between two adjacent scores, choose the "
        "lower score when the uncertainty concerns preservation "
        "of source meaning; otherwise choose the score that best "
        "matches the rubric definitions."
    ),
]


# ------------------------------------------------------------
# 4. DIRECTION INSTRUCTIONS
# ------------------------------------------------------------

DIRECTION_GUIDANCE = {

    "en-zh": {
        "source_language": "English",
        "target_language": "Mandarin Chinese",
        "instruction": (
            "Evaluate whether the Mandarin Chinese candidate "
            "faithfully translates the English source."
        ),
    },

    "zh-en": {
        "source_language": "Mandarin Chinese",
        "target_language": "English",
        "instruction": (
            "Evaluate whether the English candidate faithfully "
            "translates the Mandarin Chinese source."
        ),
    },
}


# ------------------------------------------------------------
# 5. COMMENT POLICY
# ------------------------------------------------------------
#
# Comments are optional for ordinary examples but strongly
# encouraged for low-scoring or ambiguous cases.

COMMENT_POLICY = {

    "required_for_scores": [0, 1],

    "recommended_for_scores": [2],

    "optional_for_scores": [3, 4],

    "instruction": (
        "For scores 0 or 1, briefly identify the main reason "
        "for the low score. For score 2, a short explanation is "
        "recommended. Comments for scores 3 and 4 are optional."
    ),
}


# ------------------------------------------------------------
# 6. ERROR COMMENT TAGS
# ------------------------------------------------------------
#
# Reviewers do NOT have to use these exact tags, but they are
# provided to make comments more consistent.

SUGGESTED_ERROR_TAGS = {

    "omission": (
        "Important source information is missing."
    ),

    "addition": (
        "The candidate adds unsupported information."
    ),

    "mistranslation": (
        "Source content is translated with the wrong meaning."
    ),

    "wrong_language": (
        "Output is primarily in the wrong target language."
    ),

    "source_copy": (
        "The source was copied rather than translated."
    ),

    "fluency": (
        "Target-language wording is difficult, unnatural, "
        "or grammatically problematic."
    ),

    "entity_or_number": (
        "A person, place, object, date, number, quantity, "
        "or other important detail is incorrect."
    ),

    "polarity": (
        "Negation or affirmation is reversed or distorted."
    ),

    "unrelated": (
        "Candidate meaning is largely unrelated to the source."
    ),
}


# ------------------------------------------------------------
# 7. COMPLETE REVIEWER INSTRUCTIONS
# ------------------------------------------------------------

REVIEWER_INSTRUCTIONS = f"""
MANDARIN TRANSLATION HUMAN-REVIEW INSTRUCTIONS

PURPOSE
-------
You are evaluating translation quality for a bilingual
English/Mandarin dataset.

Each row contains:

1. Review ID
2. Example ID
3. Translation direction
4. Source text
5. Candidate translation
6. Reference translation
7. Human score
8. Reviewer comment


WHAT TO SCORE
-------------
Evaluate the CANDIDATE TRANSLATION against the SOURCE TEXT.

The goal is to determine how successfully the candidate
translates the source into the required target language.

For EN -> ZH:
    Source language: English
    Target language: Mandarin Chinese

For ZH -> EN:
    Source language: Mandarin Chinese
    Target language: English


IMPORTANT: REFERENCE TRANSLATION
--------------------------------
The reference translation is provided only as an aid.

Do NOT assume that the reference is perfect.

The candidate does NOT need to use the same wording as the
reference.

If the candidate is a faithful and natural translation of the
source, it can receive the highest score even when it differs
from the reference.


SCORING SCALE
-------------
4 - EXCELLENT

The candidate accurately preserves the source meaning and is
natural and understandable in the target language.

No meaningful translation error is present.

Minor stylistic or punctuation differences that do not affect
quality are acceptable.


3 - GOOD

The main source meaning is preserved.

There may be a minor mistranslation, omission, awkward phrase,
grammar issue, or fluency problem, but the central meaning
remains correct and understandable.


2 - FAIR

The candidate communicates part or most of the source meaning,
but contains a noticeable translation problem.

Examples include a meaningful omission, addition,
mistranslation, or fluency problem.

The translation remains partially useful.


1 - POOR

The candidate preserves only a limited portion of the source
meaning or contains major translation errors.

Important information is missing, incorrect, or seriously
distorted.


0 - FAILED

The candidate is unusable as a translation of the source.

Examples include:

- wrong-language output
- copying the source instead of translating it
- unrelated output
- nearly complete loss of source meaning
- severe nonsense


WHAT TO CONSIDER
----------------
Consider the following when choosing the overall score:

1. Meaning preservation
2. Completeness
3. Target-language quality
4. Meaning-changing errors

Pay particular attention to:

- omissions
- unsupported additions
- incorrect entities
- incorrect numbers or quantities
- incorrect actions or relationships
- negation/polarity errors
- wrong-language output
- source copying


SCORING RULES
-------------
Use exactly ONE integer score:

    0, 1, 2, 3, or 4

Do not use:

    3.5
    2.5
    percentages
    letter grades

Judge the source meaning, not word-for-word similarity to the
reference.

Do not penalize a valid alternative translation simply because
it differs from the reference.

Do not reward a candidate merely because it resembles the
reference if it fails to translate the source correctly.


COMMENTS
--------
For scores 0 or 1:
    A brief reviewer comment is REQUIRED.

For score 2:
    A brief reviewer comment is RECOMMENDED.

For scores 3 or 4:
    A comment is OPTIONAL.

Useful comment descriptions include:

    omission
    addition
    mistranslation
    wrong language
    source copy
    fluency
    entity/number error
    polarity error
    unrelated output


FINAL CHECK BEFORE SUBMISSION
-----------------------------
For every row:

1. Read the source.
2. Read the candidate translation.
3. Use the reference only as supporting context.
4. Select one score from 0 to 4.
5. Add a comment when required.
6. Continue until every row has a score.
""".strip()


# ------------------------------------------------------------
# 8. MACHINE-READABLE RUBRIC
# ------------------------------------------------------------

HUMAN_REVIEW_RUBRIC = {

    "rubric_name": (
        "Mandarin Translation Human Review Rubric"
    ),

    "rubric_version": "1.0",

    "dataset_name": DATASET_NAME,

    "dataset_version": DATASET_VERSION,

    "score_min": SCORE_MIN,

    "score_max": SCORE_MAX,

    "allowed_scores": ALLOWED_SCORES,

    "score_type": "integer",

    "primary_judgment": (
        "candidate_translation_against_source_text"
    ),

    "reference_role": (
        "supporting_context_not_infallible_ground_truth"
    ),

    "evaluation_dimensions": (
        EVALUATION_DIMENSIONS
    ),

    "scoring_rubric": (
        SCORING_RUBRIC
    ),

    "decision_rules": (
        DECISION_RULES
    ),

    "direction_guidance": (
        DIRECTION_GUIDANCE
    ),

    "comment_policy": (
        COMMENT_POLICY
    ),

    "suggested_error_tags": (
        SUGGESTED_ERROR_TAGS
    ),
}


# ------------------------------------------------------------
# 9. VALIDATE RUBRIC
# ------------------------------------------------------------

assert set(
    SCORING_RUBRIC.keys()
) == set(ALLOWED_SCORES)

assert SCORE_MIN == 0
assert SCORE_MAX == 4

assert HUMAN_REVIEW_RUBRIC[
    "score_type"
] == "integer"

assert HUMAN_REVIEW_RUBRIC[
    "reference_role"
] == (
    "supporting_context_not_infallible_ground_truth"
)

print("\n✓ Score range defined: 0-4")
print("✓ Integer-only scoring required")
print("✓ Five score levels defined")
print("✓ Evaluation dimensions defined")
print("✓ Direction-specific guidance defined")
print("✓ Reference role explicitly defined")
print("✓ Comment policy defined")
print("✓ Suggested error tags defined")


# ------------------------------------------------------------
# 10. DISPLAY RUBRIC SUMMARY
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("SCORING RUBRIC")
print("-" * 70)

for score in sorted(
    SCORING_RUBRIC.keys(),
    reverse=True
):

    entry = SCORING_RUBRIC[score]

    print(
        f"\n{score} - "
        f"{entry['label'].upper()}"
    )

    print(
        entry["definition"]
    )


# ------------------------------------------------------------
# 11. DISPLAY DIRECTION GUIDANCE
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("DIRECTION GUIDANCE")
print("-" * 70)

for direction, info in (
    DIRECTION_GUIDANCE.items()
):

    print(
        f"\n{direction}:"
    )

    print(
        f"  Source: "
        f"{info['source_language']}"
    )

    print(
        f"  Target: "
        f"{info['target_language']}"
    )

    print(
        f"  Rule:   "
        f"{info['instruction']}"
    )


# ------------------------------------------------------------
# 12. DISPLAY COMMENT POLICY
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("COMMENT POLICY")
print("-" * 70)

print(
    "Required for scores: "
    f"{COMMENT_POLICY['required_for_scores']}"
)

print(
    "Recommended for score: "
    f"{COMMENT_POLICY['recommended_for_scores']}"
)

print(
    "Optional for scores: "
    f"{COMMENT_POLICY['optional_for_scores']}"
)


# ------------------------------------------------------------
# 13. UPDATE TASK CONFIG
# ------------------------------------------------------------

TASK_CONFIG[
    "human_review_rubric"
] = HUMAN_REVIEW_RUBRIC

TASK_CONFIG[
    "reviewer_instructions_defined"
] = True


# ------------------------------------------------------------
# 14. ACCEPTANCE-CRITERIA CHECK
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("TASK A ACCEPTANCE-CRITERIA STATUS")
print("-" * 70)

acceptance_status = {

    "additional_examples_selected": (
        len(reviewer_dataset)
        == TOTAL_TARGET
    ),

    "independent_source_used": (
        SOURCE_CONFIG[
            "original_borrowed_source"
        ]
        is False
    ),

    "range_of_quality_levels_created": (
        len(QUALITY_BUCKETS) == 4
    ),

    "provenance_recorded": (
        len(internal_metadata)
        == TOTAL_TARGET
    ),

    "target_set_upfront": (
        TARGET_PER_DIRECTION == 100
    ),

    "reviewer_instructions_defined": True,

    "scoring_rubric_defined": True,

    "reviewer_data_blinded": True,
}


for criterion, passed in (
    acceptance_status.items()
):

    symbol = "✓" if passed else "✗"

    print(
        f"{symbol} "
        f"{criterion}"
    )


assert all(
    acceptance_status.values()
)


TASK_CONFIG[
    "acceptance_criteria_status"
] = acceptance_status


# ------------------------------------------------------------
# 15. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("HUMAN-REVIEW RUBRIC SUMMARY")
print("-" * 70)

print(
    "✓ Reviewers judge candidate against source"
)

print(
    "✓ Reference is supporting context, "
    "not assumed ground truth"
)

print(
    "✓ Overall translation quality scored 0-4"
)

print(
    "✓ Integer-only scores"
)

print(
    "✓ Meaning preservation prioritized"
)

print(
    "✓ Completeness and fluency considered"
)

print(
    "✓ Critical meaning errors explicitly covered"
)

print(
    "✓ Low-score comment requirements defined"
)

print(
    "✓ Reviewer instructions are self-contained"
)

print(
    "✓ Task A acceptance criteria currently satisfied"
)

print("\nNEXT:")

print(
    "Cell 9 will export the reviewer package, "
    "internal traceability file, rubric/instructions, "
    "and manifest with hashes."
)



✓ Score range defined: 0-4
✓ Integer-only scoring required
✓ Five score levels defined
✓ Evaluation dimensions defined
✓ Direction-specific guidance defined
✓ Reference role explicitly defined
✓ Comment policy defined
✓ Suggested error tags defined

----------------------------------------------------------------------
SCORING RUBRIC
----------------------------------------------------------------------

4 - EXCELLENT
The candidate accurately preserves the source meaning and is natural and understandable in the target language. No meaningful translation error is present. Very small stylistic or punctuation differences that do not affect quality are allowed.

3 - GOOD
The candidate preserves the main source meaning and is understandable, but contains a minor error or awkwardness. The issue does not materially change the central meaning.

2 - FAIR
The candidate communicates part or most of the source meaning, but contains a noticeable error, omission, addition, mistranslation, or fluenc

In [18]:
# EXPORT + HASH FINAL TASK A HUMAN-REVIEW PACKAGE


print("\n" + "=" * 70)
print("STEP 9: EXPORT FINAL TASK A HUMAN-REVIEW PACKAGE")
print("=" * 70)

import os
import csv
import json
import hashlib
from datetime import datetime, timezone
from collections import Counter



REQUIRED_OBJECTS = [
    "reviewer_dataset",
    "internal_metadata",
    "HUMAN_REVIEW_RUBRIC",
    "REVIEWER_INSTRUCTIONS",
    "TASK_CONFIG",
]

missing_objects = [
    name
    for name in REQUIRED_OBJECTS
    if name not in globals()
]

assert not missing_objects, (
    "Missing required objects: "
    + ", ".join(missing_objects)
)

assert len(reviewer_dataset) == TOTAL_TARGET
assert len(internal_metadata) == TOTAL_TARGET

print("✓ Required Task A objects found")
print(f"✓ Reviewer rows: {len(reviewer_dataset)}")
print(f"✓ Internal rows: {len(internal_metadata)}")



FINAL_PACKAGE_DIR = os.path.join(
    OUTPUT_DIR,
    "final_package"
)

os.makedirs(
    FINAL_PACKAGE_DIR,
    exist_ok=True
)

print(
    f"\nFinal package directory:\n"
    f"  {FINAL_PACKAGE_DIR}"
)


# ------------------------------------------------------------
# 3. FILE PATHS
# ------------------------------------------------------------

REVIEWER_CSV_PATH = os.path.join(
    FINAL_PACKAGE_DIR,
    "mandarin_human_review_reviewer.csv"
)

REVIEWER_JSONL_PATH = os.path.join(
    FINAL_PACKAGE_DIR,
    "mandarin_human_review_reviewer.jsonl"
)

INTERNAL_CSV_PATH = os.path.join(
    FINAL_PACKAGE_DIR,
    "mandarin_human_review_internal_metadata.csv"
)

INTERNAL_JSONL_PATH = os.path.join(
    FINAL_PACKAGE_DIR,
    "mandarin_human_review_internal_metadata.jsonl"
)

RUBRIC_JSON_PATH = os.path.join(
    FINAL_PACKAGE_DIR,
    "human_review_rubric.json"
)

INSTRUCTIONS_PATH = os.path.join(
    FINAL_PACKAGE_DIR,
    "reviewer_instructions.txt"
)

TASK_CONFIG_PATH = os.path.join(
    FINAL_PACKAGE_DIR,
    "task_config.json"
)

MANIFEST_PATH = os.path.join(
    FINAL_PACKAGE_DIR,
    "task_a_manifest.json"
)


# ------------------------------------------------------------
# 4. GENERIC EXPORT HELPERS
# ------------------------------------------------------------

def write_csv(path, rows, fieldnames):

    with open(
        path,
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
            extrasaction="ignore"
        )

        writer.writeheader()

        for row in rows:
            writer.writerow(row)


def write_jsonl(path, rows):

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        for row in rows:

            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                    sort_keys=True
                )
                + "\n"
            )


def write_json(path, obj):

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            obj,
            f,
            ensure_ascii=False,
            indent=2,
            sort_keys=True
        )


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


# ------------------------------------------------------------
# 5. FINAL REVIEWER EXPORT SCHEMA
# ------------------------------------------------------------

FINAL_REVIEWER_FIELDS = [
    "review_id",
    "example_id",
    "direction",
    "source_text",
    "candidate_translation",
    "reference_translation",
    "human_score",
    "reviewer_comment",
]


assert set(FINAL_REVIEWER_FIELDS) == set(
    REVIEWER_FIELDS
)


# ------------------------------------------------------------
# 6. FINAL INTERNAL EXPORT SCHEMA
# ------------------------------------------------------------

FINAL_INTERNAL_FIELDS = [
    "review_id",
    "example_id",
    "direction",
    "source_language",
    "target_language",
    "source_dataset",
    "source_config",
    "source_split",
    "source_record_id",
    "original_source_text",
    "original_reference_text",
    "reference_translation",
    "candidate_translation",
    "internal_quality_bucket",
    "candidate_generation_method",
    "dataset_version",
]




FORBIDDEN_REVIEWER_EXPORT_FIELDS = {
    "internal_quality_bucket",
    "candidate_generation_method",
    "source_dataset",
    "source_config",
    "source_split",
    "source_record_id",
    "dataset_version",
    "quality_seed",
}


for row in reviewer_dataset:

    leaked = (
        FORBIDDEN_REVIEWER_EXPORT_FIELDS
        .intersection(row.keys())
    )

    assert not leaked, (
        f"Internal metadata leak detected: {leaked}"
    )


print(
    "\n✓ Reviewer export passed "
    "internal-metadata leak check"
)


# ------------------------------------------------------------
# 8. VALIDATE BLANK HUMAN LABELS
# ------------------------------------------------------------

assert all(
    row["human_score"] == ""
    for row in reviewer_dataset
)

assert all(
    row["reviewer_comment"] == ""
    for row in reviewer_dataset
)

print(
    "✓ Human labels remain blank before export"
)


# ------------------------------------------------------------
# 9. EXPORT REVIEWER DATA
# ------------------------------------------------------------

write_csv(
    REVIEWER_CSV_PATH,
    reviewer_dataset,
    FINAL_REVIEWER_FIELDS
)

write_jsonl(
    REVIEWER_JSONL_PATH,
    reviewer_dataset
)

print("\n✓ Reviewer CSV exported")
print("✓ Reviewer JSONL exported")


# ------------------------------------------------------------
# 10. EXPORT INTERNAL TRACEABILITY DATA
# ------------------------------------------------------------

write_csv(
    INTERNAL_CSV_PATH,
    internal_metadata,
    FINAL_INTERNAL_FIELDS
)

write_jsonl(
    INTERNAL_JSONL_PATH,
    internal_metadata
)

print("✓ Internal metadata CSV exported")
print("✓ Internal metadata JSONL exported")


# ------------------------------------------------------------
# 11. EXPORT RUBRIC
# ------------------------------------------------------------

write_json(
    RUBRIC_JSON_PATH,
    HUMAN_REVIEW_RUBRIC
)

print("✓ Machine-readable rubric exported")


# ------------------------------------------------------------
# 12. EXPORT HUMAN-READABLE INSTRUCTIONS
# ------------------------------------------------------------

with open(
    INSTRUCTIONS_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        REVIEWER_INSTRUCTIONS
    )

    f.write("\n")

print("✓ Reviewer instructions exported")


# ------------------------------------------------------------
# 13. UPDATE CONFIG BEFORE EXPORT
# ------------------------------------------------------------

TASK_CONFIG[
    "final_package"
] = {
    "reviewer_csv": os.path.basename(
        REVIEWER_CSV_PATH
    ),
    "reviewer_jsonl": os.path.basename(
        REVIEWER_JSONL_PATH
    ),
    "internal_csv": os.path.basename(
        INTERNAL_CSV_PATH
    ),
    "internal_jsonl": os.path.basename(
        INTERNAL_JSONL_PATH
    ),
    "rubric_json": os.path.basename(
        RUBRIC_JSON_PATH
    ),
    "reviewer_instructions": os.path.basename(
        INSTRUCTIONS_PATH
    ),
}


write_json(
    TASK_CONFIG_PATH,
    TASK_CONFIG
)

print("✓ Task configuration exported")


# ------------------------------------------------------------
# 14. RELOAD REVIEWER CSV AND VERIFY
# ------------------------------------------------------------

with open(
    REVIEWER_CSV_PATH,
    "r",
    encoding="utf-8-sig",
    newline=""
) as f:

    reloaded_reviewer = list(
        csv.DictReader(f)
    )


assert (
    len(reloaded_reviewer)
    == TOTAL_TARGET
)

assert (
    list(reloaded_reviewer[0].keys())
    == FINAL_REVIEWER_FIELDS
)

assert all(
    row["human_score"] == ""
    for row in reloaded_reviewer
)

assert all(
    row["reviewer_comment"] == ""
    for row in reloaded_reviewer
)

print(
    "\n✓ Reviewer CSV reload validation passed"
)


# ------------------------------------------------------------
# 15. RELOAD INTERNAL CSV AND VERIFY
# ------------------------------------------------------------

with open(
    INTERNAL_CSV_PATH,
    "r",
    encoding="utf-8-sig",
    newline=""
) as f:

    reloaded_internal = list(
        csv.DictReader(f)
    )


assert (
    len(reloaded_internal)
    == TOTAL_TARGET
)

assert (
    list(reloaded_internal[0].keys())
    == FINAL_INTERNAL_FIELDS
)

print(
    "✓ Internal CSV reload validation passed"
)


# ------------------------------------------------------------
# 16. VERIFY REVIEW/INTERNAL MAPPING AFTER EXPORT
# ------------------------------------------------------------

reloaded_internal_by_id = {
    row["review_id"]: row
    for row in reloaded_internal
}


for reviewer_row in reloaded_reviewer:

    review_id = reviewer_row[
        "review_id"
    ]

    assert (
        review_id
        in reloaded_internal_by_id
    )

    internal_row = (
        reloaded_internal_by_id[
            review_id
        ]
    )

    assert (
        reviewer_row["example_id"]
        == internal_row["example_id"]
    )

    assert (
        reviewer_row["candidate_translation"]
        == internal_row["candidate_translation"]
    )


print(
    "✓ Exported reviewer/internal mapping validated"
)


# ------------------------------------------------------------
# 17. VERIFY DIRECTION COUNTS AFTER EXPORT
# ------------------------------------------------------------

export_direction_counts = Counter(
    row["direction"]
    for row in reloaded_reviewer
)


assert (
    export_direction_counts["en-zh"]
    == TARGET_PER_DIRECTION
)

assert (
    export_direction_counts["zh-en"]
    == TARGET_PER_DIRECTION
)

print(
    "✓ Exported direction counts validated"
)


# ------------------------------------------------------------
# 18. VERIFY INTERNAL QUALITY COUNTS AFTER EXPORT
# ------------------------------------------------------------

export_quality_counts = Counter(
    (
        row["direction"],
        row["internal_quality_bucket"]
    )
    for row in reloaded_internal
)


for direction in DIRECTIONS:

    for bucket in QUALITY_BUCKETS:

        assert (
            export_quality_counts[
                (direction, bucket)
            ]
            == QUALITY_PER_BUCKET_PER_DIRECTION
        )


print(
    "✓ Exported internal quality distribution validated"
)


# ------------------------------------------------------------
# 19. HASH ALL PRIMARY FILES
# ------------------------------------------------------------

PRIMARY_FILES = [
    REVIEWER_CSV_PATH,
    REVIEWER_JSONL_PATH,
    INTERNAL_CSV_PATH,
    INTERNAL_JSONL_PATH,
    RUBRIC_JSON_PATH,
    INSTRUCTIONS_PATH,
    TASK_CONFIG_PATH,
]


FILE_HASHES = {
    os.path.basename(path): sha256_file(path)
    for path in PRIMARY_FILES
}

print("\n" + "-" * 70)
print("SHA-256 FILE HASHES")
print("-" * 70)

for filename, digest in FILE_HASHES.items():

    print(
        f"{filename}\n"
        f"  {digest}"
    )


# ------------------------------------------------------------
# 20. BUILD FINAL MANIFEST
# ------------------------------------------------------------

manifest_created_utc = (
    datetime.now(timezone.utc)
    .replace(microsecond=0)
    .isoformat()
)


TASK_A_MANIFEST = {

    "task": (
        "Task A: Prepare A Mandarin Dataset "
        "For Human Review"
    ),

    "dataset_name": DATASET_NAME,

    "dataset_version": DATASET_VERSION,

    "created_utc": manifest_created_utc,

    "random_seed": SEED,

    "source": {
        "dataset": SOURCE_CONFIG[
            "dataset_name"
        ],
        "config": SOURCE_CONFIG[
            "dataset_config"
        ],
        "split": SOURCE_SPLIT,
        "source_type": SOURCE_CONFIG[
            "source_type"
        ],
    },

    "excluded_sources": (
        sorted(EXCLUDED_SOURCES)
    ),

    "targets": {
        "en-zh": TARGET_PER_DIRECTION,
        "zh-en": TARGET_PER_DIRECTION,
        "total": TOTAL_TARGET,
    },

    "actual_counts": {
        "en-zh": (
            export_direction_counts[
                "en-zh"
            ]
        ),
        "zh-en": (
            export_direction_counts[
                "zh-en"
            ]
        ),
        "total": len(
            reloaded_reviewer
        ),
    },

    "internal_quality_distribution": {
        direction: {
            bucket: (
                export_quality_counts[
                    (direction, bucket)
                ]
            )
            for bucket in QUALITY_BUCKETS
        }
        for direction in DIRECTIONS
    },

    "quality_bucket_role": (
        "internal_sampling_and_construction_only"
    ),

    "human_score_role": (
        "authoritative_fresh_review_label"
    ),

    "human_score_scale": {
        "minimum": SCORE_MIN,
        "maximum": SCORE_MAX,
        "allowed": ALLOWED_SCORES,
        "type": "integer",
    },

    "reviewer_blinding": {
        "internal_quality_bucket_hidden": True,
        "candidate_generation_method_hidden": True,
        "source_provenance_hidden": True,
    },

    "traceability": {
        "source_dataset_recorded": True,
        "source_split_recorded": True,
        "source_record_id_recorded": True,
        "original_bilingual_pair_recorded": True,
        "candidate_generation_method_recorded": True,
    },

    "separation_policy": {
        "reuse_finetuning_examples": False,
        "reuse_existing_judge_calibration_examples": False,
        "reuse_prompt_validation_examples": False,
        "reuse_sealed_final_test_examples": False,
    },

    "acceptance_criteria": (
        acceptance_status
    ),

    "files": {
        filename: {
            "sha256": digest
        }
        for filename, digest
        in FILE_HASHES.items()
    },
}


# ------------------------------------------------------------
# 21. WRITE MANIFEST
# ------------------------------------------------------------

write_json(
    MANIFEST_PATH,
    TASK_A_MANIFEST
)

MANIFEST_HASH = sha256_file(
    MANIFEST_PATH
)

print(
    "\n✓ Final manifest exported"
)

print(
    "Manifest SHA-256:\n"
    f"  {MANIFEST_HASH}"
)


# ------------------------------------------------------------
# 22. VERIFY MANIFEST
# ------------------------------------------------------------

with open(
    MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as f:

    reloaded_manifest = json.load(f)


assert (
    reloaded_manifest[
        "actual_counts"
    ]["total"]
    == TOTAL_TARGET
)

assert (
    reloaded_manifest[
        "actual_counts"
    ]["en-zh"]
    == TARGET_PER_DIRECTION
)

assert (
    reloaded_manifest[
        "actual_counts"
    ]["zh-en"]
    == TARGET_PER_DIRECTION
)

assert all(
    reloaded_manifest[
        "acceptance_criteria"
    ].values()
)

print(
    "✓ Manifest reload validation passed"
)


# ------------------------------------------------------------
# 23. VERIFY HASHES FROM DISK
# ------------------------------------------------------------

for path in PRIMARY_FILES:

    filename = os.path.basename(
        path
    )

    expected = FILE_HASHES[
        filename
    ]

    actual = sha256_file(
        path
    )

    assert actual == expected


print(
    "✓ All primary file hashes verified"
)


# ------------------------------------------------------------
# 24. PACKAGE INVENTORY
# ------------------------------------------------------------

PACKAGE_FILES = (
    PRIMARY_FILES
    + [MANIFEST_PATH]
)

print("\n" + "-" * 70)
print("FINAL PACKAGE INVENTORY")
print("-" * 70)

for path in PACKAGE_FILES:

    size_bytes = os.path.getsize(
        path
    )

    print(
        f"✓ {os.path.basename(path):45} "
        f"{size_bytes:>10,} bytes"
    )


# ------------------------------------------------------------
# 25. FINAL ACCEPTANCE CHECK
# ------------------------------------------------------------

FINAL_CHECKS = {

    "200_reviewer_examples": (
        len(reloaded_reviewer)
        == TOTAL_TARGET
    ),

    "100_en_zh": (
        export_direction_counts["en-zh"]
        == TARGET_PER_DIRECTION
    ),

    "100_zh_en": (
        export_direction_counts["zh-en"]
        == TARGET_PER_DIRECTION
    ),

    "independent_source": (
        SOURCE_CONFIG[
            "original_borrowed_source"
        ]
        is False
    ),

    "traceability_exported": (
        len(reloaded_internal)
        == TOTAL_TARGET
    ),

    "quality_range_present": (
        len(export_quality_counts)
        == (
            len(DIRECTIONS)
            * len(QUALITY_BUCKETS)
        )
    ),

    "rubric_exported": (
        os.path.exists(
            RUBRIC_JSON_PATH
        )
    ),

    "instructions_exported": (
        os.path.exists(
            INSTRUCTIONS_PATH
        )
    ),

    "manifest_exported": (
        os.path.exists(
            MANIFEST_PATH
        )
    ),

    "reviewer_data_blinded": True,

    "human_scores_blank": all(
        row["human_score"] == ""
        for row in reloaded_reviewer
    ),
}


print("\n" + "-" * 70)
print("FINAL TASK A VALIDATION")
print("-" * 70)

for check, passed in FINAL_CHECKS.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{check}"
    )


assert all(
    FINAL_CHECKS.values()
)


# ------------------------------------------------------------
# 26. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TASK A PACKAGE COMPLETE")
print("=" * 70)

print(
    f"\nDataset version: "
    f"{DATASET_VERSION}"
)

print(
    f"Reviewer examples: "
    f"{len(reloaded_reviewer)}"
)

print(
    f"  EN -> ZH: "
    f"{export_direction_counts['en-zh']}"
)

print(
    f"  ZH -> EN: "
    f"{export_direction_counts['zh-en']}"
)

print(
    "\nHuman scores: BLANK "
    "(ready for fresh human review)"
)

print(
    "\nReviewer-facing files:"
)

print(
    "  - "
    + os.path.basename(
        REVIEWER_CSV_PATH
    )
)

print(
    "  - "
    + os.path.basename(
        REVIEWER_JSONL_PATH
    )
)

print(
    "  - "
    + os.path.basename(
        INSTRUCTIONS_PATH
    )
)

print(
    "  - "
    + os.path.basename(
        RUBRIC_JSON_PATH
    )
)

print(
    "\nINTERNAL ONLY:"
)

print(
    "  - "
    + os.path.basename(
        INTERNAL_CSV_PATH
    )
)

print(
    "  - "
    + os.path.basename(
        INTERNAL_JSONL_PATH
    )
)

print(
    "  - "
    + os.path.basename(
        TASK_CONFIG_PATH
    )
)

print(
    "  - "
    + os.path.basename(
        MANIFEST_PATH
    )
)

print(
    "\nIMPORTANT:"
)

print(
    "Do NOT send internal_metadata files "
    "to human reviewers."
)

print(
    "Send the reviewer dataset together with "
    "reviewer_instructions.txt."
)

print(
    "\nFinal package directory:"
)

print(
    FINAL_PACKAGE_DIR
)



STEP 9: EXPORT FINAL TASK A HUMAN-REVIEW PACKAGE
✓ Required Task A objects found
✓ Reviewer rows: 200
✓ Internal rows: 200

Final package directory:
  task_a_human_review/final_package

✓ Reviewer export passed internal-metadata leak check
✓ Human labels remain blank before export

✓ Reviewer CSV exported
✓ Reviewer JSONL exported
✓ Internal metadata CSV exported
✓ Internal metadata JSONL exported
✓ Machine-readable rubric exported
✓ Reviewer instructions exported
✓ Task configuration exported

✓ Reviewer CSV reload validation passed
✓ Internal CSV reload validation passed
✓ Exported reviewer/internal mapping validated
✓ Exported direction counts validated
✓ Exported internal quality distribution validated

----------------------------------------------------------------------
SHA-256 FILE HASHES
----------------------------------------------------------------------
mandarin_human_review_reviewer.csv
  eb5bdc56a431a0f288e77d04c7dbe0c52caa52aa8750a8cd1d2c8f8d1e6b7309
mandarin_human_rev

In [20]:
import shutil
from google.colab import files

# 1. Zip the target folder (replace 'my_folder' with your actual folder path)
shutil.make_archive('task_a_human_review', 'zip', 'task_a_human_review')

# 2. Download the created zip file
files.download('task_a_human_review.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>